# ARC-AGI-3: the arc agent

Generated by `arc_agent/kaggle/build.py`; do not edit here.

In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
import os
FILES = {'__init__.py': '', 'agent.py': '"""The arc_agent, as one game-playing object: `ArcAgent(game_id).act(frame_data) -> (action id, data)`.\n\nA thin driver: it hands each transition to the brain, tells the executive whether a WIN happened, asks the executive\nfor the next action, and keeps the engine\'s rules (RESET after GAME_OVER; never RESET before acting in a level, since\nthe engine would restart the whole game).\n"""\nimport json\nimport math\nimport os\nimport random\nfrom collections import Counter\n\nimport numpy as np\n\nfrom arc_agent.blueprint import DEFAULT\nfrom arc_agent.brain import Brain, size_bucket\nfrom arc_agent.executive import Executive\nfrom arc_agent.overhypothesis import Overhypothesis\n\nRESET, CLICK = 0, 6\nPRIORS = os.path.join(os.path.dirname(os.path.abspath(__file__)), "priors.json")\n\n\ndef shipped_priors():\n    """What warm-up play on other games taught (warmup.py), if it was shipped: goal-type draws, expert-kind draws, and\n    the formula library\'s templates."""\n    try:\n        return json.load(open(PRIORS))\n    except (OSError, ValueError):\n        return {}\n\n\ndef kind_boost(draws):\n    """Part 23: an overhypothesis over the kinds of attended experts; a kind\'s head start is log(P(kind) x kinds)."""\n    from arc_agent.column import feature_kind  # noqa: F401  (the kinds\' vocabulary lives there)\n    kinds = ("unconditional", "ahead", "elsewhere", "other column", "own state", "quotient", "numeric", "conjunction")\n    if not draws:\n        return {}\n    o = Overhypothesis()\n    for d in draws:\n        o.add("kind", d)\n    p = o.prior("kind", kinds)\n    return {k: math.log(p[k] * len(kinds)) for k in kinds}\n\n\ndef library_specs(specs):\n    """Formula templates as the function finder takes them (from JSON, factors arrive as lists: make them tuples)."""\n    return [dict(s, theta=np.asarray(s["theta"], float), free=np.asarray(s["free"], bool),\n                 factors=tuple(tuple(f) for f in s["factors"])) for s in specs or []]\n\n\ndef region_size(grid, x, y, cap=200):\n    c, seen, q = grid[y, x], {(x, y)}, [(x, y)]\n    while q and len(seen) < cap:\n        px, py = q.pop()\n        for nx, ny in ((px + 1, py), (px - 1, py), (px, py + 1), (px, py - 1)):\n            if (nx, ny) not in seen and 0 <= nx < grid.shape[1] and 0 <= ny < grid.shape[0] and grid[ny, nx] == c:\n                seen.add((nx, ny))\n                q.append((nx, ny))\n    return len(seen)\n\n\nclass ArcAgent:\n    def __init__(self, game_id="", blueprint=DEFAULT, seed=0, priors=None):\n        """priors: {"draws": goal types, "kinds": expert kinds, "library": formula templates}; None = shipped."""\n        self.game_id = game_id\n        pri = shipped_priors() if priors is None else priors\n        self.brain = Brain(blueprint, kind_prior=kind_boost(pri.get("kinds", [])),\n                           library=library_specs(pri.get("library", [])))\n        self.executive = Executive(self.brain, pri.get("draws", []))\n        self.level, self.prev, self.acted = None, None, 0\n        self.expected = None\n        self.rng = random.Random(seed)\n        self.notes = Counter()\n\n    def kind(self, action, grid):\n        aid, x, y = action\n        if aid != CLICK:\n            return ("key", aid)\n        return ("click", int(grid[y, x]), size_bucket(region_size(grid, x, y)))\n\n    def act(self, fd):\n        state = getattr(fd.state, "name", str(fd.state))\n        grid = np.array(fd.frame[-1], dtype=np.int8, copy=True) if len(fd.frame) else None\n        if state in ("NOT_PLAYED", "GAME_OVER") or grid is None:\n            self.prev, self.acted, self.executive.plan = None, 0, []\n            self.notes["reset (game over)" if state == "GAME_OVER" else "reset (start)"] += 1\n            return RESET, None\n        if self.level != fd.levels_completed:          # a new level: the last action won the previous one\n            if self.prev is not None and self.level is not None and fd.levels_completed > self.level:\n                pgrid, paction, _ = self.prev\n                states, _ = self.brain.model_step(dict(self.brain.joint()), paction, self.brain.static_scene())\n                self.executive.observe(win_states=states)\n                self.notes["win"] += 1\n            self.level = fd.levels_completed\n            self.executive.since = 0\n            self.brain.start(grid, new_level=True)\n            self.executive.plan, self.prev, self.acted = [], None, 0\n            self.executive.start_grid = grid.copy()\n        elif self.prev is None:                         # the level restarted\n            self.brain.start(grid, new_level=False)\n            self.executive.plan = []\n        else:\n            pgrid, paction, pkind = self.prev\n            report = self.brain.observe(pgrid, paction, grid, pkind)\n            self.executive.observe()\n            if report["recruited"] or report["lost"]:\n                self.executive.plan = []\n            else:\n                self.executive.surprise(self.expected)\n            self.notes["recruited"] += len(report["recruited"])\n        self.executive.can_reset = self.acted > 0         # the engine restarts the whole game otherwise\n        choice = self.executive.decide(list(fd.available_actions))\n        if choice is not None and choice[0][0] == RESET:  # a plan that uses RESET (its effect is in the model)\n            self.prev, self.acted, self.executive.plan = None, 0, []\n            self.notes["reset (planned)"] += 1\n            return RESET, None\n        if choice is None:\n            choice = self.executive.fallback(list(fd.available_actions))\n            if choice is not None:\n                self.notes["fallback"] += 1\n        if choice is None:\n            avail = [a for a in fd.available_actions if a not in (RESET, CLICK)]\n            if avail:\n                action = (self.rng.choice(avail), None, None)\n            else:\n                ys, xs = np.nonzero(grid != self.brain.v1.ground)\n                i = self.rng.randrange(len(xs)) if len(xs) else 0\n                action = (CLICK, int(xs[i]) if len(xs) else 32, int(ys[i]) if len(ys) else 32)\n            self.expected = None\n            self.notes["random"] += 1\n        else:\n            action, self.expected = choice\n            self.notes[self.executive.kind_of_plan] += 1\n        self.prev = (grid, action, self.kind(action, grid))\n        self.acted += 1\n        self.executive.since += 1\n        aid, x, y = action\n        return aid, ({"x": x, "y": y} if aid == CLICK else None)\n', 'blueprint.py': '"""The brain blueprint (ARCHITECTURE.md section 2): which areas the brain has, how many columns each holds, and which\nreference frame they use. Innate structure lives here (priors are geometries); learned areas get only a menu of frame\ntypes and find their own frames from the data. `Brain(blueprint)` builds the columns. Adding an area, a frame type or\n(later) a PFC is an edit here, not in the brain.\n"""\nfrom dataclasses import dataclass, field\n\n\n@dataclass(frozen=True)\nclass Area:\n    name: str\n    role: str                     # "visual" | "motor" | "association" | "episodic" | "executive"\n    columns: int                  # how many column slots\n    frame: str                    # innate frame name, or "learned"\n    frame_types: tuple = ()       # for learned areas: the frame types a column may take\n    inputs: tuple = ()            # names of the areas that feed this one\n    params: dict = field(default_factory=dict)\n\n\n@dataclass(frozen=True)\nclass Blueprint:\n    areas: tuple\n\n    def area(self, name):\n        return next(a for a in self.areas if a.name == name)\n\n    def by_role(self, role):\n        return [a for a in self.areas if a.role == role]\n\n\nDEFAULT = Blueprint(areas=(\n    Area("V1", "visual", 1, frame="retina", params=dict(height=64, width=64)),\n    Area("M1", "motor", 1, frame="motor_map", inputs=("V1",)),\n    Area("assoc", "association", 32, frame="learned", frame_types=("translation", "recolour", "bar"),\n         inputs=("V1", "M1")),\n    Area("HC", "episodic", 1, frame="product", inputs=("assoc",)),\n    Area("PFC", "executive", 0, frame="router", inputs=("assoc", "HC")),     # a stand-in for now: executive.py\n))\n', 'brain.py': '"""The brain (ARCHITECTURE.md sections 2 and 5): built from a blueprint; one perception-and-learning step per action.\n\nAreas: V1 (innate retina: the scene, the ground colour, and a Spotlight lattice of what lies under things), M1 (innate\nmotor map: keys, and click proposals aligned with V1, ranked by what has responded), assoc (learned columns, each a\nmodel over its own frame), HC (episodes in a Spotlight lattice addressed by the joint state).\n\nOne step: every assoc column is looked for where its model says it went; what each action did to it is learned, in\nthe context the column read in its own frame before the action (part 20: conditions by attention, no rule says what\nblocks); the change no column explains is split into sites, and frame types compete to explain each (translation by a\nHough vote over displacements, recolour by a consistent colour map, bar by extending a counter); a free slot takes the\nwinner.\n"""\nimport math\nfrom collections import Counter\n\nimport numpy as np\n\nfrom arc_agent.blueprint import DEFAULT\nfrom arc_agent.column import PMIN, Column\nfrom arc_agent.frames import H, W, Bar, Recolour, Translation\nfrom arc_agent.hidden import Hidden\nfrom arc_agent.pfc import PFC\nfrom arc_agent.spotlight import Spotlight\n\nRADIUS = 16                       # the largest displacement looked for in one step\nFRAC = 0.75                       # a column is still itself when this share of its cells match (structure persists)\nMAX_SITES = 24                    # more change sites than this in one step is a scene change, not motion\n\n\ndef components(mask, conn8=True):\n    seen = np.zeros(mask.shape, bool)\n    out = []\n    nb = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if conn8 else [])\n    for y, x in zip(*np.nonzero(mask)):\n        if seen[y, x]:\n            continue\n        comp, q = [], [(y, x)]\n        seen[y, x] = True\n        while q:\n            cy, cx = q.pop()\n            comp.append((cx, cy))\n            for dy, dx in nb:\n                ny, nx = cy + dy, cx + dx\n                if 0 <= ny < mask.shape[0] and 0 <= nx < mask.shape[1] and mask[ny, nx] and not seen[ny, nx]:\n                    seen[ny, nx] = True\n                    q.append((ny, nx))\n        out.append(comp)\n    return out\n\n\ndef regions(grid):\n    """4-connected same-colour regions: [(colour, [(x, y)])]."""\n    out = []\n    seen = np.zeros(grid.shape, bool)\n    for y in range(grid.shape[0]):\n        for x in range(grid.shape[1]):\n            if seen[y, x]:\n                continue\n            c, comp, q = grid[y, x], [], [(y, x)]\n            seen[y, x] = True\n            while q:\n                cy, cx = q.pop()\n                comp.append((cx, cy))\n                for ny, nx in ((cy + 1, cx), (cy - 1, cx), (cy, cx + 1), (cy, cx - 1)):\n                    if 0 <= ny < grid.shape[0] and 0 <= nx < grid.shape[1] and not seen[ny, nx] and grid[ny, nx] == c:\n                        seen[ny, nx] = True\n                        q.append((ny, nx))\n            out.append((int(c), comp))\n    return out\n\n\ndef size_bucket(n):\n    return 0 if n <= 2 else 1 if n <= 9 else 2 if n <= 36 else 3 if n <= 144 else 4\n\n\nclass LazyContext:\n    """A column\'s context computed on demand (for planning): the same features as Brain.context, but each is read only\n    when an expert asks for it. Conjunction features ("and", f, v, g) are present when f\'s value is v."""\n\n    def __init__(self, brain, col, state, scene, others, under):\n        self.brain = brain\n        self.col, self.state, self.scene, self.others, self.under = col, state, scene, others, under\n        self.window = brain.window_set(col)\n        self.cells = col.frame.place(state)\n        self.anchor = self.cells[0]\n        self.own, self.vals = None, {}\n\n    def __contains__(self, f):\n        if f[0] == "at":\n            return (f[1], f[2]) in self.window\n        if f[0] == "self":\n            return True\n        if f[0] == "col":\n            return f[1] in self.others and f[1] != self.col.cid\n        if f[0] == "and":\n            return f[1] in self and f[3] in self and self.get(f[1]) == f[2]\n        if f[0] in ("mod", "num"):\n            return self.col.frame.kind == "translation"\n        return False\n\n    def get(self, f, default=None):\n        if f is None:                                   # the unconditional expert reads nothing\n            return default\n        if f in self.vals:\n            return self.vals[f]\n        if f[0] == "at":\n            x, y = int(self.anchor[0]) + f[1], int(self.anchor[1]) + f[2]\n            if not (0 <= x < W and 0 <= y < H):\n                v = -1\n            else:\n                if self.own is None:\n                    self.own = set(map(tuple, self.cells.tolist()))\n                v = self.under(x, y) if (x, y) in self.own else int(self.scene[y, x])\n        elif f[0] == "self":\n            v = self.state\n        elif f[0] == "col":\n            v = self.others.get(f[1], default)\n        elif f[0] == "and":\n            v = self.get(f[3], default)\n        elif f[0] == "mod":\n            v = self.brain.quotients(self.col, self.state)[f]\n        elif f[0] == "num":\n            v = self.brain.numeric(self.col, self.state, self.scene, self.under)\n        else:\n            v = default\n        self.vals[f] = v\n        return v\n\n\nclass V1:\n    def __init__(self):\n        self.scene, self.ground = None, 0\n        self.under = Spotlight()                        # (x, y) -> "under": the colour beneath a column\n\n    def start(self, grid, new_level):\n        self.scene = grid.copy()\n        if new_level:\n            v, n = np.unique(grid, return_counts=True)\n            self.ground = int(v[np.argmax(n)])\n            self.under.forget()\n\n    def under_at(self, x, y):\n        return self.under.value((x, y), "under", self.ground)\n\n\nclass M1:\n    """Click proposals: one point per V1 region (a first guess, demoted by evidence), and P(an action of this kind\n    changes anything), Beta(1, 1), for keys and for clicks by the clicked region\'s signature (colour, size)."""\n\n    def __init__(self):\n        self.effect = {}\n        self.done = set()                               # click addresses tried in this level\n\n    def tried(self, action):\n        return action[0] == 6 and (action[1], action[2]) in self.done\n\n    def p_effect(self, kind):\n        e, n = self.effect.get(kind, (0, 0))\n        return (e + 1) / (n + 2)\n\n    def record(self, kind, effect):\n        e, n = self.effect.get(kind, (0, 0))\n        self.effect[kind] = (e + bool(effect), n + 1)\n\n    def proposals(self, grid, ground, claimed):\n        out = []\n        for c, cells in regions(grid):\n            if c == ground and len(cells) > 64:\n                continue\n            if all(claimed[y, x] for x, y in cells):\n                continue\n            xs, ys = zip(*cells)\n            cx, cy = np.mean(xs), np.mean(ys)\n            x, y = min(cells, key=lambda p: (p[0] - cx) ** 2 + (p[1] - cy) ** 2)\n            out.append((int(x), int(y), ("click", c, size_bucket(len(cells)))))\n        return out\n\n\nclass Brain:\n    def __init__(self, blueprint=DEFAULT, conditional=True, kind_prior=None, library=None, pfc=None, hidden=True):\n        self.bp, self.conditional = blueprint, conditional\n        self.pfc = pfc if pfc is not None else PFC()    # part 24: the prefrontal column, shared with whoever passed it\n        self.hidden = Hidden(conditional, kind_prior) if hidden else None   # part 27: columns not on the screen\n        self.level, self.last_act = 0, "key"\n        self.kind_prior, self.library = kind_prior or {}, library or []\n        self.slots = blueprint.area("assoc").columns\n        self.v1, self.m1, self.hc = V1(), M1(), Spotlight()\n        self.columns, self.next_id, self.t = [], 0, 0\n        self.last_kind = None\n        self.cache = {}                                 # planning: contexts per joint state\n        self.last_evidence, self.last_cells = {}, {}     # part 24: the last step\'s same-cell evidence per candidate                           # M1\'s kind of the last action\n\n    # ---- the joint state and the columns used for planning ------------------------------------------------------\n    def active(self):\n        return [c for c in self.columns if c.active]\n\n    def planning(self):\n        """The columns planned with: action-driven, not a counter, and, if they compete with others to explain the same\n        change, the one the PFC holds responsible (part 24)."""\n        return [c for c in self.columns if c.active and c.frame.kind != "bar" and c.driver() == "action"\n                and self.pfc.responsible(c.cid, self.last_act)]\n\n    def clocks(self):\n        return [c for c in self.columns if c.active and (c.frame.kind == "bar" or c.driver() == "clock")]\n\n    def joint(self):\n        return tuple((c.cid, c.state) for c in self.planning())\n\n    def others(self, exclude=None, states=None):\n        """The other columns\' states a column reads as context: the planning columns\' (or the given joint states), and\n        the hidden-state columns\' (part 27)."""\n        out = {c.cid: c.state for c in self.planning() if c is not exclude} if states is None else dict(states)\n        if self.hidden is not None:\n            out.update(self.hidden.states())\n        return out\n\n    def claimed(self):\n        m = np.zeros((H, W), bool)\n        for c in self.clocks():\n            p = c.frame.place(c.state)\n            ok = (p[:, 0] >= 0) & (p[:, 0] < W) & (p[:, 1] >= 0) & (p[:, 1] < H)\n            m[p[ok, 1], p[ok, 0]] = True\n        return m\n\n    def modelled(self, action):\n        """Some planning column has seen this action (so its effect is the columns\' to predict, not M1\'s)."""\n        return any(c.belief(c.akey(action)).n > 0 for c in self.planning())\n\n    # ---- context, read by a column in its own frame (part 20) ----------------------------------------------------\n    def window(self, col):\n        """Positions around the column, relative to its anchor (its first cell), out to its reach."""\n        key = (len(col.frame.cells0), col.reach)\n        if getattr(col, "_window", (None,))[0] != key:\n            rel = col.frame.cells0 - col.frame.cells0[0]\n            (x0, y0), (x1, y1) = rel.min(0), rel.max(0)\n            m = col.reach\n            area = (x1 - x0 + 2 * m + 1) * (y1 - y0 + 2 * m + 1)\n            step = max(1, int(math.ceil(math.sqrt(area / 400))))\n            xs, ys = np.arange(x0 - m, x1 + m + 1, step), np.arange(y0 - m, y1 + m + 1, step)\n            col._window = (key, np.array([(x, y) for y in ys for x in xs], int))\n        return col._window[1]\n\n    def window_set(self, col):\n        offs = self.window(col)\n        if getattr(col, "_window_set", (None,))[0] is not col._window:\n            col._window_set = (col._window, set(map(tuple, offs.tolist())))\n        return col._window_set[1]\n\n    def context(self, col, state, scene, others, under):\n        """The column\'s features: ("at", dx, dy) the content at that position relative to its anchor (beneath the\n        column: what lies under it; off the screen: -1); ("self",) its own state; ("col", cid) another column\'s\n        state."""\n        offs = self.window(col)\n        cells = col.frame.place(state)\n        pts = offs + cells[0]\n        ok = (pts[:, 0] >= 0) & (pts[:, 0] < W) & (pts[:, 1] >= 0) & (pts[:, 1] < H)\n        vals = np.full(len(pts), -1, int)\n        vals[ok] = scene[pts[ok, 1], pts[ok, 0]]\n        own = set(map(tuple, cells.tolist()))\n        for i, (x, y) in enumerate(pts.tolist()):\n            if ok[i] and (x, y) in own:\n                vals[i] = under(x, y)\n        ctx = {("at", int(dx), int(dy)): int(v) for (dx, dy), v in zip(offs.tolist(), vals.tolist())}\n        ctx[("self",)] = state\n        for cid, st in others.items():\n            if cid != col.cid:\n                ctx[("col", cid)] = st\n        if col.frame.kind == "translation":\n            ctx.update(self.quotients(col, state))\n            ctx[("num",)] = self.numeric(col, state, scene, under)\n        return ctx\n\n    def quotients(self, col, state):\n        """Part 10: content regularity is a quotient of the geometry. The column\'s position modulo k, in units of its\n        own moves."""\n        u = col.unit or 1\n        return {("mod", k): ((state[0] // u) % k, (state[1] // u) % k) for k in range(2, 7)}\n\n    def numeric(self, col, state, scene, under):\n        """The numeric features (part 23): along each of four rays from the column\'s middle, the distance to the\n        first cell unlike what lies beneath the column (or to the screen\'s edge); then its own position."""\n        cells = col.frame.place(state)\n        (x0, y0), (x1, y1) = cells.min(0), cells.max(0)\n        cx, cy = (x0 + x1) // 2, (y0 + y1) // 2\n        base = under(int(cells[0][0]), int(cells[0][1])) if 0 <= cells[0][0] < W and 0 <= cells[0][1] < H else -1\n        out = []\n        for (dx, dy), (sx, sy) in (((0, -1), (cx, y0)), ((0, 1), (cx, y1)), ((-1, 0), (x0, cy)), ((1, 0), (x1, cy))):\n            d, x, y = 0, sx + dx, sy + dy\n            while 0 <= x < W and 0 <= y < H and scene[y, x] == base:\n                d, x, y = d + 1, x + dx, y + dy\n            out.append(d)\n        return tuple(out) + (int(state[0]), int(state[1]))\n\n    # ---- a new level, or the same level restarted ---------------------------------------------------------------\n    def start(self, grid, new_level):\n        self.v1.start(grid, new_level)\n        if self.hidden is not None:\n            self.hidden.start(new_level)\n        if new_level:\n            self.m1.done.clear()\n            self.level += 1\n        for c in self.columns:\n            f = c.frame\n            if f.kind == "translation":\n                hits = f.find_all(grid)\n                c.active = bool(hits)\n                if hits:\n                    c.state = min(hits, key=lambda s: abs(s[0] - c.state[0]) + abs(s[1] - c.state[1]))\n            elif f.kind == "recolour":\n                c.state, c.active = f.read(grid), True\n            else:\n                c.state = f.read(grid)\n\n    # ---- one step ---------------------------------------------------------------------------------------------\n    def observe(self, before, action, after, kind=None):\n        """Learn from one transition. Returns a report: which columns moved, how many cells stayed unexplained, and\n        which columns were recruited."""\n        self.t += 1\n        self.last_kind = kind\n        self.last_act = "key" if action[0] != 6 else "click"\n        ground = self.v1.ground\n        report = dict(moved=[], lost=[], recruited=[], unexplained=0)\n        render = before.copy()\n        others = self.others()\n        learned = []                                    # part 27: what each visible column saw, for the hidden-state area\n        planned = set(others)                           # (only the models in use: the PFC\'s rivals are not evidence)\n        ctxs = {c.cid: self.context(c, c.state, before, others, self.v1.under_at) for c in self.active()}\n        painted, claimed = {}, {}                       # part 24: each grouped column\'s prediction of its cells\n        for c in self.active():\n            if c.cid in self.pfc.member:\n                painted[c.cid], claimed[c.cid] = self._paint(c, action, ctxs[c.cid])\n        for c in self.active():\n            f, old = c.frame, c.state\n            akey = c.akey(action, old)\n            ctx = ctxs[c.cid]\n            if f.kind == "translation":\n                p, _ = c.dist(akey, ctx)\n                guess = f.apply(old, max(p, key=p.get))\n                new = guess if f.matches(after, guess) else f.find(after, old, RADIUS)\n                if new is None:                         # part of its content changed: the best partial match nearby\n                    new = guess if f.score(after, guess) >= FRAC else f.find_near(after, old, RADIUS, FRAC)\n                    if new is not None:                 # rewrite the changed cells in its content memory\n                        p = f.place(new)\n                        seen = after[p[:, 1], p[:, 0]]\n                        for i in np.nonzero(seen != f.colours)[0]:\n                            f.colours[i] = seen[i]\n                            c.content.write(int(i), "colour", int(seen[i]))\n                if new is None:                         # not nearby: anywhere on the screen\n                    hits = f.find_all(after)\n                    new = min(hits, key=lambda h: abs(h[0] - old[0]) + abs(h[1] - old[1])) if hits else None\n                if new is None:\n                    c.active = False\n                    report["lost"].append(c.cid)\n                    continue\n                op = f.op_between(old, new)\n                c.learn(akey, ctx, op, self.t)\n                learned.append(((c.cid, akey), c.belief(akey)))\n                cells_old = f.place(old)\n                selfset = set(map(tuple, cells_old.tolist()))\n                for x, y in cells_old.tolist():\n                    if 0 <= x < W and 0 <= y < H:\n                        render[y, x] = after[y, x]       # vacated cells show what was beneath\n                newset = set(map(tuple, f.place(new).tolist()))\n                for x, y in newset - selfset:            # newly covered: remember what was beneath (permanence)\n                    if 0 <= x < W and 0 <= y < H:\n                        self.v1.under.write((x, y), "under", int(before[y, x]))\n                self.v1.under.forget(selfset - newset)\n                f.draw(render, new)\n                c.state = new\n            else:\n                new = f.read(after)\n                op = f.op_between(old, new)\n                if op is None:\n                    c.active = False\n                    report["lost"].append(c.cid)\n                    continue\n                c.learn(akey, ctx, op, self.t)\n                learned.append(((c.cid, akey), c.belief(akey)))\n                f.draw(render, new)\n                c.state = new\n            if op != f.identity:\n                report["moved"].append(c.cid)\n        if self.hidden is not None and self.hidden.observe(self.t, action[0], kind if kind is not None\n                                                           else ("key", action[0]),\n                                                           [(m, b) for m, b in learned\n                                                            if b.conditional and b.T and m[0] in planned]):\n            self.restate()\n        residual = render != after\n        report["unexplained"] = int(residual.sum())\n        self.pfc.observe(self._group_evidence(painted, claimed, before, after), self.last_act)   # part 24\n        if residual.any():\n            self._recruit(before, after, residual, action, report)\n        # M1: did this action change anything beyond the clocks?\n        clockset = set(c.cid for c in self.clocks())\n        effect = bool([m for m in report["moved"] if m not in clockset]) or bool(report["recruited"])\n        if self.last_kind is not None:\n            self.m1.record(self.last_kind, effect)\n        if action[0] == 6:\n            self.m1.done.add((action[1], action[2]))\n        self.v1.scene = after.copy()\n        return report\n\n    def restate(self):\n        """After the hidden graph is refitted: rewrite its state in every stored episode that reads it."""\n        g = self.hidden.graph_col.cid\n        for c in self.columns + self.hidden.columns():\n            for b in c.keys.values():\n                b.restate(("self",) if c.cid == g else ("col", g), self.hidden.value_at)\n\n    # ---- recruitment: frame types compete to explain what is left -----------------------------------------------\n    def _new(self, frame, state):\n        if len([c for c in self.columns if c.active]) >= self.slots:\n            idle = [c for c in self.columns if not c.active]\n            if not idle:\n                return None\n            self.columns.remove(min(idle, key=lambda c: c.uses))\n        c = Column(self.next_id, frame, state, self.t, self.conditional, self.kind_prior, self.library)\n        self.next_id += 1\n        self.columns.append(c)\n        return c\n\n    def _recruit(self, before, after, residual, action, report):\n        ground = self.v1.ground\n        sites = components(residual)\n        if len(sites) > MAX_SITES:\n            return\n        original = residual.copy()\n        rigid = []                                      # (column, the cells it explains)\n        for _ in range(3):                              # rigid explanations: common fate, translation or turn\n            found = self._translation(before, after, residual, ground)\n            turned = self._rotation(before, after, residual, ground)\n            if turned is not None and (found is None or turned[2] > self._explained(found[0], found[1], residual)):\n                found = turned[:2]\n            if found is None:\n                break\n            cells, d = found\n            d = tuple(d) + (0,) * (3 - len(d))\n            frame = Translation(cells, [before[y, x] for x, y in cells])\n            col = self._new(frame, (0, 0, 0))\n            if col is None:\n                return\n            akey = col.akey(action, (0, 0, 0))\n            others = self.others(col)\n            col.learn(akey, self.context(col, (0, 0, 0), before, others, self.v1.under_at), d)   # evidence (part 15)\n            col.state = d\n            covered = set(map(tuple, cells))\n            for x, y in cells:\n                residual[y, x] = False\n            for x, y in frame.place(d).tolist():\n                if 0 <= x < W and 0 <= y < H:\n                    residual[y, x] = False\n                    covered.add((x, y))\n                    if (x, y) not in set(map(tuple, cells)):\n                        self.v1.under.write((x, y), "under", int(before[y, x]))\n            rigid.append((col, covered))\n            report["recruited"].append(col.cid)\n        # every change with one colour map is also explained as a recolour (part 24: no fixed order between frames).\n        # A rigid explanation\'s rival is ONE recolour over all the changed cells it covers (both strips of a toggle,\n        # not each strip alone), so the two are compared on the same change; the PFC decides between them.\n        sites = components(original)\n        used = set()\n        for r, cov in rigid:\n            idx = [k for k, site in enumerate(sites) if cov & set(site)]\n            if not idx:\n                continue\n            used |= set(idx)\n            union = [cell for k in idx for cell in sites[k]]\n            col = self._recolour(union, before, after, action, report)\n            if col is not None:\n                region = set(map(tuple, r.frame.cells0.tolist())) | set(union)\n                self.pfc.group([r.cid, col.cid], [r.frame.kind, "recolour"], self.region_context(region, before, after))\n        for k, site in enumerate(sites):                # the rest: a recolour, or one more cell of a counter\n            if k in used:\n                continue\n            pairs = [(int(before[y, x]), int(after[y, x])) for x, y in site]\n            m = {}\n            if any(m.setdefault(a, b) != b for a, b in pairs) or self._extend_bar(site, pairs):\n                continue\n            self._recolour(site, before, after, action, report)\n\n    def _recolour(self, cells, before, after, action, report):\n        """A recolour column over these cells, if one colour map explains their change; else None."""\n        pairs = [(int(before[y, x]), int(after[y, x])) for x, y in cells]\n        m = {}\n        if any(m.setdefault(a, b) != b for a, b in pairs):\n            return None\n        frame = Recolour(cells, [a for a, _ in pairs])\n        col = self._new(frame, tuple(a for a, _ in pairs))\n        if col is None:\n            return None\n        op = frame.op_between(col.state, tuple(b for _, b in pairs))\n        others = self.others(col)\n        col.learn(col.akey(action, col.state), self.context(col, col.state, before, others, self.v1.under_at), op)\n        col.state = tuple(b for _, b in pairs)\n        report["recruited"].append(col.cid)\n        return col\n\n    def _paint(self, c, action, ctx):\n        """What column c predicts for the cells it claims this step: {cell: colour}, and the cells it claims (where it\n        is now and where it predicts it will be)."""\n        f, s = c.frame, c.state\n        p, _ = c.dist(c.akey(action, s), ctx)\n        ns = f.apply(s, max(p, key=p.get))\n        out = {}\n        if f.kind == "translation":\n            for x, y in f.place(s).tolist():\n                out[(x, y)] = self.v1.under_at(x, y)     # what it leaves shows what lay beneath\n            for (x, y), v in zip(f.place(ns).tolist(), f.colours.tolist()):\n                out[(x, y)] = int(v)\n        elif f.kind == "recolour":\n            for (x, y), v in zip(f.place(ns).tolist(), ns):\n                out[(x, y)] = int(v)\n        return out, set(out)\n\n    def _group_evidence(self, painted, claimed, before, after):\n        """Part 24: the rivals in a group are scored on the SAME cells, every cell any of them claims. Each predicts\n        the cells it doesn\'t model as unchanged. Returns {cid: log predictive probability of what was seen}."""\n        lp, hit, miss = {}, math.log(0.99), math.log(0.01)\n        self.last_cells = {}\n        for g in self.pfc.groups.values():\n            members = [c for c in g["members"] if c in painted]\n            cells = set()\n            for c in members:\n                cells |= claimed[c]\n            cells = [(x, y) for x, y in cells if 0 <= x < W and 0 <= y < H]\n            for c in members:\n                n = sum(1 for x, y in cells if painted[c].get((x, y), int(before[y, x])) == int(after[y, x]))\n                lp[c] = n * hit + (len(cells) - n) * miss\n                self.last_cells[c] = len(cells)\n        self.last_evidence = lp\n        return lp\n\n    def region_context(self, region, before, after):\n        """The PFC\'s game-agnostic context for a contested region (part 24)."""\n        xs, ys = zip(*region)\n        colours = {int(before[y, x]) for x, y in region} | {int(after[y, x]) for x, y in region}\n        return {("size",): size_bucket(len(region)), ("colours",): min(4, len(colours)),\n                ("thin",): min(max(xs) - min(xs), max(ys) - min(ys)) <= 2, ("level",): min(self.level, 2)}\n\n    @staticmethod\n    def _explained(cells, d, residual):\n        f = Translation(cells, [0] * len(cells))\n        d = tuple(d) + (0,) * (3 - len(d))\n        pts = set(map(tuple, f.place((0, 0, 0)).tolist())) | set(map(tuple, f.place(d).tolist()))\n        return sum(1 for x, y in pts if 0 <= x < W and 0 <= y < H and residual[y, x])\n\n    def _rotation(self, before, after, residual, ground):\n        """Part 12: a set of cells turned about its own pivot (and perhaps moved). Candidate sets: the connected\n        non-ground regions of the old screen that the change touches. Returns (cells, (dx, dy, k), explained)."""\n        mask = before != ground\n        best = None\n        for comp in components(mask):\n            if not 3 <= len(comp) <= 300 or not any(residual[y, x] for x, y in comp):\n                continue\n            f = Translation(comp, [before[y, x] for x, y in comp])\n            for k in (1, 2, 3):\n                for dx in range(-4, 5):\n                    for dy in range(-4, 5):\n                        s = (dx, dy, k)\n                        if f.matches(after, s):\n                            e = self._explained(comp, s, residual)\n                            if best is None or e > best[2]:\n                                best = (comp, s, e)\n        return best\n\n    def harvest(self):\n        """What the columns learned that carries to other games (part 23): the kinds of their attended experts, and\n        the formulas the numeric experts fitted."""\n        kinds, formulas = [], []\n        for c in self.columns:\n            kinds += c.harvest()\n            for b in c.keys.values():\n                if b.numeric is not None:\n                    formulas += [(r, len(b.numeric.X)) for r in b.numeric.formulas()]\n        return kinds, formulas\n\n    def _extend_bar(self, site, pairs):\n        """A small change with one colour map next to a column that just made the same change: a counter\'s next cell.\n        A one-site recolour column becomes a bar; a bar grows."""\n        if len(site) > 16 or len({p for p in pairs}) != 1:\n            return False\n        a, b = pairs[0]\n        for c in self.active():\n            f = c.frame\n            if f.kind == "bar" and (f.empty, f.full) == (a, b):\n                near = f.cells0\n            elif f.kind == "recolour" and len(set(f.colours.tolist())) == 1 and int(f.colours[0]) == a and \\\n                    c.state and all(v == b for v in c.state):\n                near = f.cells0\n            else:\n                continue\n            if min(max(abs(x - nx), abs(y - ny)) for x, y in site for nx, ny in near.tolist()) > 3:\n                continue\n            if f.kind == "recolour":                      # becomes a bar: its cells, then the new ones\n                bar = Bar(list(map(tuple, f.cells0.tolist())) + list(site), a, b)\n                c.frame, c.state = bar, len(f.cells0) + len(site)\n                c.keys, c.ev = {}, {"action": 0.0, "clock": 0.0}\n                c.clock = type(c.clock)(bar.K, bar.identity, 0.5)\n            else:\n                for cell in site:\n                    f.extend(cell)\n                c.state += len(site)\n            return True\n        return False\n\n    def _translation(self, before, after, residual, ground):\n        """The displacement most changed cells agree on (old colour at p, the same colour at p + d), and the rigid set\n        that moved by it: grown from the changed cells through same-coloured neighbours that also moved by d, minus\n        unchanged cells nothing covers after the move."""\n        ys, xs = np.nonzero(residual)\n        old = [(x, y, before[y, x]) for x, y in zip(xs, ys) if before[y, x] != ground]\n        new = {}\n        for x, y in zip(xs, ys):\n            if after[y, x] != ground:\n                new.setdefault(int(after[y, x]), []).append((x, y))\n        if not old or not new:\n            return None\n        votes, voters = Counter(), {}\n        by_colour = {}\n        for x, y, c in old:\n            by_colour.setdefault(int(c), []).append((x, y))\n        for c, pts in by_colour.items():\n            if c not in new:\n                continue\n            a = np.array(pts)\n            b = np.array(new[c])\n            d = (b[None, :, :] - a[:, None, :]).reshape(-1, 2)\n            d = d[(np.abs(d[:, 0]) <= RADIUS) & (np.abs(d[:, 1]) <= RADIUS) & ((d[:, 0] != 0) | (d[:, 1] != 0))]\n            for k, n in zip(*np.unique(d, axis=0, return_counts=True)):\n                votes[(int(k[0]), int(k[1]))] += int(n)\n                voters.setdefault((int(k[0]), int(k[1])), set()).add(c)\n        if not votes:\n            return None\n        best = None\n        for d, v in sorted(votes.items(), key=lambda kv: (-kv[1], abs(kv[0][0]) + abs(kv[0][1])))[:3]:\n            if v < 2:\n                break\n            M = self._moved_set(before, after, residual, ground, [o for o in old if int(o[2]) in voters[d]], d)\n            if not M:\n                continue\n            moved_to = {(x + d[0], y + d[1]) for x, y in M}\n            explained = sum(1 for x, y in M | moved_to if 0 <= x < W and 0 <= y < H and residual[y, x])\n            # figure and ground: a boundary drawn through one colour costs (a uniform region\'s motion is not\n            # identifiable), so "a floor patch moved" loses to "the thing on the floor moved"\n            cut = sum(1 for x, y in M for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1))\n                      if (nx, ny) not in M and 0 <= nx < W and 0 <= ny < H and before[ny, nx] == before[y, x])\n            score = explained - cut\n            if best is None or score > best[0]:\n                best = (score, M, d)\n        if best is None or best[0] <= 0:\n            return None\n        cells = sorted(best[1], key=lambda p: (p[1], p[0]))\n        return cells, best[2]\n\n    def _moved_set(self, before, after, residual, ground, old, d):\n        """The rigid set that moved by d: grown from the changed cells through same-coloured neighbours that also\n        moved by d, minus unchanged cells that nothing covers after the move."""\n        def moved(x, y):\n            nx, ny = x + d[0], y + d[1]\n            return 0 <= nx < W and 0 <= ny < H and before[y, x] != ground and after[ny, nx] == before[y, x]\n\n        seed = [(x, y) for x, y, _ in old if moved(x, y)]\n        cols = {int(before[y, x]) for x, y in seed}\n        M, q = set(seed), list(seed)\n        while q and len(M) < 600:\n            x, y = q.pop()\n            for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):\n                if (nx, ny) not in M and 0 <= nx < W and 0 <= ny < H and int(before[ny, nx]) in cols and moved(nx, ny):\n                    M.add((nx, ny))\n                    q.append((nx, ny))\n        changed = True\n        while changed:                                  # an unchanged cell must be covered by the moved set\n            changed = False\n            for x, y in list(M):\n                if not residual[y, x] and before[y, x] == after[y, x]:\n                    src = (x - d[0], y - d[1])\n                    if src not in M or before[src[1], src[0]] != after[y, x]:\n                        M.discard((x, y))\n                        changed = True\n        return M\n\n    # ---- the model, for planning ------------------------------------------------------------------------------\n    def static_scene(self):\n        """The scene with every planning column taken out: what lies beneath them, or the ground."""\n        g = self.v1.scene.copy()\n        for c in self.planning():\n            for x, y in c.frame.place(c.state).tolist():\n                if 0 <= x < W and 0 <= y < H:\n                    g[y, x] = self.v1.under_at(x, y)\n        return g\n\n    def model_step(self, states, action, static):\n        """Predicted next states of the planning columns ({cid: state}) and the bits an uncertain outcome is worth\n        (part 18: the entropy of the predicted OUTCOME). bits > 0 means the outcome here is not known."""\n        cols = {c.cid: c for c in self.planning()}\n        key = tuple(sorted(states.items()))\n        if key not in self.cache:                       # the scene and every column\'s context at this joint state\n            scene = static.copy()\n            for cid, st in states.items():\n                cols[cid].frame.draw(scene, st)\n            others = self.others(states=states)\n            self.cache[key] = {cid: LazyContext(self, cols[cid], st, scene, others, lambda x, y: int(static[y, x]))\n                               for cid, st in states.items()}\n        ctxs = self.cache[key]\n        new, bits = dict(states), 0.0\n        for cid, s in states.items():\n            c = cols[cid]\n            akey = c.akey(action, s)\n            if akey[0] == "at":                         # a click elsewhere: unconditional, so cached per search\n                ukey = ("u", cid, akey)\n                if ukey not in self.cache:\n                    self.cache[ukey] = c.dist(akey, {})[0]\n                p = self.cache[ukey]\n                if p.get(c.frame.identity, 0.0) >= 1 - PMIN:\n                    continue\n            else:\n                p, _ = c.dist(akey, ctxs[cid])\n            outcomes = {}\n            for op, q in p.items():\n                ns = c.frame.apply(s, op)\n                outcomes[ns] = outcomes.get(ns, 0.0) + q\n            rest = max(0.0, 1.0 - sum(p.values()))\n            if rest > 1e-9:\n                outcomes[("unknown", cid)] = rest       # an operator never seen: an outcome not yet imagined\n            ns, q = max(outcomes.items(), key=lambda kv: kv[1])\n            if q < 1 - PMIN:\n                bits = max(bits, min(1.0, -sum(x * math.log2(x) for x in outcomes.values() if x > 0)))\n                continue\n            new[cid] = ns\n        return new, bits\n', 'column.py': '"""An assoc column (ARCHITECTURE.md section 3): a reusable model over one reference frame.\n\n- frame: the coordinate system and its operators (frames.py);\n- state: where the column is in its frame now;\n- content: its template, in a Spotlight lattice addressed by the frame\'s own coordinates (the cell index), kept apart\n  from the structure (the offsets);\n- what each action does to it, conditioned on context by in-context attention (parts 26, 27): every observation is\n  kept as an episode; keys are context features read in the column\'s own frame (the content at one position relative\n  to it; another column\'s state; quotients; numeric features); what a key\'s value means, and how much a key is\n  trusted, are read from the episodes of similar situations, where competing situation dimensions say which episodes\n  are similar. Nothing says what blocks a move: it is whatever the attention learns to read;\n- drivers competing by evidence (part 13): "action" (the operator depends on which action) against "clock" (one\n  operator on every action, as counters and timers behave).\n\nAn action, seen from a column: a key ("key", id); a click on the column\'s own cells ("self",); a click elsewhere\n("at", x, y), whose effect on this column is a remote binding.\n"""\nimport math\nfrom collections import Counter\n\nimport numpy as np\n\nfrom arc_agent.spotlight import Spotlight\n\nA = 0.02                          # pseudo-count: one consistent observation makes a deterministic outcome ~0.98\nPMIN = 0.02\nTOP = 8                           # keys read for a prediction (the most attended)\nMEMORY, DTOP, ETA = 1000, 3, 0.1  # episodes kept per action; heads mixed (part 27); theta\'s step (part 26\'s)\nQUERY_ONLY = ("self",)            # the column\'s own state is never an item to read\nQUERY_KINDS = ("col", "self")     # the situation (with what lies beneath): other columns\' states, the own state.\n                                  # Not quotients: five codes of one position, whose similarity weights add up\nNUM_MIN, NUM_EVERY = 6, 8         # the numeric expert: observations before its first fit; new ones between refits\n\n\ndef feature_kind(f, op=None):\n    """What kind of context an expert reads, relative to the operator when it is a move (for priors across games)."""\n    if f is None:\n        return "unconditional"\n    if f[0] == "at":\n        if isinstance(op, tuple) and len(op) == 3 and all(isinstance(v, int) for v in op) and (op[0] or op[1]):\n            return "ahead" if f[1] * op[0] + f[2] * op[1] > 0 else "elsewhere"\n        return "elsewhere"\n    return {"col": "other column", "self": "own state", "mod": "quotient", "num": "numeric",\n            "and": "conjunction"}.get(f[0], "other")\n\n\nclass Numeric:\n    """The numeric expert (part 23): a translation\'s displacement as a function of the column\'s numeric features\n    (distances along four rays, its own position), fitted by the function finder of experiments A-B3\n    (symreg.py: residual-greedy, affine atoms, BIC). Refitted when its own prediction and the\n    column\'s attention-weighted prediction both fail (part 25: compute is spent where error remains), less often\n    after each refit that leaves the recent data unexplained."""\n\n    def __init__(self, library=None):\n        self.X, self.Y, self.fit, self.last, self.library = [], [], None, -10 ** 9, library or []\n        self.every = NUM_EVERY                          # doubles after each refit that doesn\'t explain the data\n\n    def predict(self, x):\n        if self.fit is None:\n            return None\n        out = []\n        for f in self.fit:\n            if isinstance(f, int):\n                out.append(f)\n                continue\n            with np.errstate(all="ignore"):\n                v = float(f.predict(np.array([x], float))[0])\n            if not math.isfinite(v):                    # outside where the formula is defined: no prediction\n                return None\n            out.append(int(round(v)))\n        return tuple(out)\n\n    def add(self, x, op, missed=True):\n        self.X.append(tuple(float(v) for v in x))\n        self.Y.append(op)\n        if missed and len(self.X) >= NUM_MIN and len(self.X) - self.last >= self.every and self.predict(x) != op:\n            self.refit()\n            if any(self.predict(xx) != yy for xx, yy in zip(self.X[-NUM_MIN:], self.Y[-NUM_MIN:])):\n                self.every *= 2                         # compute is spent where a formula can pay off\n\n    def refit(self):\n        from arc_agent.symreg import Finder\n        X = np.array(self.X, float)\n        fit = []\n        for c in range(len(self.Y[0])):\n            y = np.array([o[c] for o in self.Y], float)\n            if np.all(y == y[0]) or c == 2:\n                fit.append(int(Counter(int(v) for v in y).most_common(1)[0][0]))\n            else:\n                fit.append(Finder([f"n{i}" for i in range(X.shape[1])], library=self.library, affine=True,\n                                  k_max=2).fit(X, y))\n        self.fit, self.last = fit, len(self.X)\n\n    def formulas(self):\n        return [f.best for f in (self.fit or []) if not isinstance(f, int)]\nP_IDENTITY = {"key": 0.5, "self": 0.5, "at": 0.99,  # innate: a click elsewhere rarely changes me (locality)\n              "pfc": 0.5}                         # the PFC column: staying with the same model, before any evidence\n\n\nclass ConditionalOps:\n    """What one action does to one column, by in-context attention with one head per situation feature (parts 26, 27,\n    27b; docs/ATTENTION.md section 7, docs/HIDDEN_STATE.md section 8).\n\n    Every observation is kept as an episode: the situation (the other columns\' states, hidden ones included; what lies\n    beneath the column; its own state), every key\'s value, the operator seen, and each key\'s prequential log probability\n    of it. A key is a context feature read in the column\'s own frame (the content at one position relative to it;\n    another column\'s state; a quotient; the numeric expert), or the unconditional expert (key None).\n\n    Heads: head 0 counts every episode as 1 (part 20\'s track record); head f counts the episodes that match now on\n    situation feature f as 1 and the others as exp(-theta_f) (part 26\'s similarity on that one feature; a large theta_f\n    is part 21\'s conjunction cases on f). Within a head, the same weights say which key is read (its prior, part 23\'s\n    description length plus the warm-up head start, plus the weighted sum of its stored log probabilities; the TOP most\n    attended are read) and what a key\'s value means (weighted counts, backing off to the head\'s weighted counts).\n    theta_f is learned by the gradient of the head\'s own log probability of each new operator (part 26\'s rule, for one\n    feature). Heads compete: each is weighted by a prior (half on head 0, the rest shared by the features) times its\n    prequential evidence, and the DTOP heaviest are mixed. A feature that does not matter takes a share of the weight\n    instead of shrinking every episode\'s (part 26\'s product over features did that, part 27\'s shared key ranking lost\n    situation-dependent reading)."""\n\n    def __init__(self, K, identity, p0, conditional=True, kind_prior=None, library=None):\n        self.K, self.identity, self.p0, self.conditional = K, identity, p0, conditional\n        self.kind_prior = kind_prior or {}              # kind -> log head start, learned on other games (part 23)\n        self.numeric = Numeric(library) if conditional else None\n        self.n0 = Counter()                             # every observation, unweighted\n        self.n = 0\n        self.keys, self.index, self.prior = [None], {None: 0}, [0.0]\n        self.sfeats, self.sindex = [], {}               # the situation\'s features: head i + 1 reads sfeats[i]\n        self.ev = np.zeros(1)                           # each head\'s prequential log evidence (0: no situation)\n        self.theta, self.rms = np.zeros(0), np.zeros(0)  # per feature: its head\'s similarity weight, and step scale\n        self.sumL = []                                  # per feature: {value code: summed log probabilities of the\n        self.codes, self.ops, self.opindex = {}, [], {}  # episodes with that value}\n        self.T = 0                                      # the memory: situation codes S, key value codes V (-1: the\n        self.S = np.full((16, 0), -1, np.int32)         # key was absent), prequential log probabilities L (counting\n        self.V = np.full((16, 1), -1, np.int32)         # every episode), operators Y, and when each was seen\n        self.L = np.zeros((16, 1))\n        self.Y = np.zeros(16, np.int32)\n        self.when = np.zeros(16, np.int64)\n        self.tot = np.zeros(1)                          # each key\'s summed log probability over every episode\n        self.last, self.last_dim, self.cache = {}, None, {}\n\n    # ---- codes ------------------------------------------------------------------------------------------------\n    def _code(self, v, add=False):\n        try:\n            hash(v)\n        except TypeError:\n            v = repr(v)\n        c = self.codes.get(v)\n        if c is None and add:\n            c = self.codes[v] = len(self.codes)\n        return -2 if c is None else c                   # -2: a value never stored, so it matches no episode\n\n    def _situation(self, ctx, add=False):\n        q = np.full(len(self.sfeats), -1, np.int32)\n        for i, f in enumerate(self.sfeats):\n            if f == ("under",):\n                if ("at", 0, 0) in ctx:\n                    q[i] = self._code(("under", ctx.get(("at", 0, 0))), add)\n            elif f in ctx:\n                q[i] = self._code(ctx.get(f), add)\n        return q\n\n    def _weights(self):\n        """Each head\'s log weight: its prior plus its evidence."""\n        F = len(self.sfeats)\n        pri = np.full(F + 1, math.log(0.5) - (math.log(F) if F else 0.0))\n        pri[0] = math.log(0.5) if F else 0.0\n        return pri + self.ev\n\n    # ---- the backoff ------------------------------------------------------------------------------------------\n    def _base(self, op):\n        return self.p0 if op == self.identity else (1 - self.p0) / (self.K - 1)\n\n    def _p0(self):\n        """Unweighted: the clock driver and clicks elsewhere, which read no context."""\n        N = sum(self.n0.values())\n        named = set(self.n0) | {self.identity}\n        p = {op: (self.n0[op] + A * self._base(op)) / (N + A) for op in named}\n        return p, A * (1 - self.p0) / (self.K - 1) / (N + A)\n\n    # ---- reading ----------------------------------------------------------------------------------------------\n    def _head(self, d, code):\n        """Head d at situation value code: each episode\'s weight, the keys in order of score, the scores, the backoff,\n        a cache of meanings, the matching episodes, and the weight of the others."""\n        hit = self.cache.get((d, code))\n        if hit is None:\n            T = self.T\n            if d == 0:\n                m, beta, kap = None, 1.0, np.ones(T)\n                s = np.array(self.prior) + self.tot\n            else:\n                beta = math.exp(-self.theta[d - 1])\n                m = self.S[:T, d - 1] == code\n                kap = np.where(m, 1.0, beta)\n                sm = self.sumL[d - 1].get(code)\n                s = np.array(self.prior) + beta * self.tot + ((1 - beta) * sm if sm is not None else 0.0)\n            cnt = np.bincount(self.Y[:T], weights=kap, minlength=len(self.ops))\n            Z = float(kap.sum())\n            p0 = {op: (cnt[i] + A * self._base(op)) / (Z + A) for i, op in enumerate(self.ops)}\n            if self.identity not in p0:\n                p0[self.identity] = A * self._base(self.identity) / (Z + A)\n            rest = A * (1 - self.p0) / (self.K - 1) / (Z + A)\n            hit = self.cache[(d, code)] = (kap, np.argsort(-s), s, p0, rest, {}, m, beta)\n        return hit\n\n    def _table(self, hit, k, v):\n        """What key k\'s value v means in this head: ({operator: p}, probability of each operator never seen)."""\n        kap, _, _, p0, rest, tables = hit[:6]\n        c = self._code(v)\n        if (k, c) not in tables:\n            mm = self.V[:self.T, k] == c\n            N = float(kap[mm].sum())\n            if N <= 0:\n                tables[(k, c)] = (p0, rest)\n            else:\n                cnt = np.bincount(self.Y[:self.T][mm], weights=kap[mm], minlength=len(self.ops))\n                p = {op: (cnt[i] + A * p0[op]) / (N + A) for i, op in enumerate(self.ops)}\n                for op, x in p0.items():\n                    if op not in p:\n                        p[op] = A * x / (N + A)\n                tables[(k, c)] = (p, A * rest / (N + A))\n        return tables[(k, c)]\n\n    def _expert(self, hit, k, ctx):\n        f = self.keys[k]\n        if f is None:\n            return hit[3], hit[4]\n        if f[0] == "num":                               # the fitted formula\n            op = self.numeric.predict(ctx.get(f))\n            if op is None:\n                return hit[3], hit[4]\n            p = {o: 0.01 * x for o, x in hit[3].items()}\n            p[op] = p.get(op, 0.0) + 0.99\n            return p, 0.01 * hit[4]\n        return self._table(hit, k, ctx.get(f))\n\n    def _readable(self, f, ctx):\n        return f is None or (f in ctx and f[0] not in QUERY_ONLY)\n\n    def _dimension(self, d, code, ctx):\n        """One head\'s prediction from its TOP most attended keys that this context can feed: (prediction, probability\n        of each operator never seen, keys read, their attention, the head)."""\n        hit = self._head(d, code)\n        picked = []\n        for k in hit[1]:\n            if self._readable(self.keys[k], ctx):\n                picked.append(int(k))\n                if len(picked) == TOP:\n                    break\n        s = hit[2][picked]\n        w = np.exp(s - s.max())\n        w /= w.sum()\n        out, r = Counter(), 0.0\n        for k, wi in zip(picked, w):\n            p, pr = self._expert(hit, k, ctx)\n            for op, x in p.items():\n                out[op] += wi * x\n            r += wi * pr\n        return out, r, picked, w, hit\n\n    def dist(self, ctx):\n        """The prediction: ({operator: p}, probability of each operator never seen). ctx is a dict of every feature, or\n        a lazy context (brain.LazyContext) that computes only the features asked for."""\n        if not self.conditional or self.T == 0:\n            return self._p0()\n        q = self._situation(ctx)\n        lw = self._weights()\n        dims = np.argsort(-lw)[:DTOP]\n        w = np.exp(lw[dims] - lw[dims].max())\n        w /= w.sum()\n        out, r = Counter(), 0.0\n        for d, wd in zip(dims, w):\n            p, pr = self._dimension(int(d), int(q[d - 1]) if d else 0, ctx)[:2]\n            for op, x in p.items():\n                out[op] += wd * x\n            r += wd * pr\n        return dict(out), r\n\n    def best(self, ctx):\n        p, _ = self.dist(ctx)\n        op = max(p, key=p.get)\n        return op, p[op]\n\n    def predictive(self, ctx, op):\n        p, rest = self.dist(ctx)\n        return p.get(op, rest)\n\n    # ---- learning ---------------------------------------------------------------------------------------------\n    def _ensure(self, ctx, op):\n        for f in list(ctx) + [("under",)]:              # the situation\'s features\n            if (f[0] in QUERY_KINDS or f == ("under",)) and f not in self.sindex:\n                self.sindex[f] = len(self.sfeats)\n                self.sfeats.append(f)\n                self.ev = np.append(self.ev, self.ev[0])  # a newcomer starts with head 0\'s evidence\n                self.theta, self.rms = np.append(self.theta, 0.0), np.append(self.rms, 0.0)\n                self.sumL.append({-1: self.tot.copy()})   # every past episode lacked it\n                self.S = np.hstack([self.S, np.full((len(self.S), 1), -1, np.int32)])\n        feats = [f for f in ctx if f[0] not in QUERY_ONLY]\n        kinds = Counter(f[0] for f in feats)\n        new = [f for f in feats if f not in self.index]\n        if new:                                         # the new keys\' past: absent, so they backed off\n            self.V = np.hstack([self.V, np.full((len(self.V), len(new)), -1, np.int32)])\n            self.L = np.hstack([self.L, np.repeat(self.L[:, :1], len(new), 1)])\n            self.tot = np.append(self.tot, np.full(len(new), self.tot[0]))\n            for sl in self.sumL:\n                for c, arr in sl.items():\n                    sl[c] = np.append(arr, np.full(len(new), arr[0]))\n        for f in new:                                   # description length: log K behind, K the keys of its kind;\n            self.index[f] = len(self.keys)              # plus the head start its kind earned on other games\n            self.keys.append(f)\n            self.prior.append(-math.log(kinds[f[0]]) + self.kind_prior.get(feature_kind(f, op), 0.0))\n        if op not in self.opindex:\n            self.opindex[op] = len(self.ops)\n            self.ops.append(op)\n        if self.T == len(self.Y):                       # grow the memory\n            self.S = np.vstack([self.S, np.full_like(self.S, -1)])\n            self.V = np.vstack([self.V, np.full_like(self.V, -1)])\n            self.L = np.vstack([self.L, np.zeros_like(self.L)])\n            self.Y = np.append(self.Y, np.zeros_like(self.Y))\n            self.when = np.append(self.when, np.zeros_like(self.when))\n\n    def _grad(self, d, code, ctx, op):\n        """d log P(op) / d theta for head d >= 1 (one feature: matching episodes count 1, the others beta)."""\n        _, _, picked, a, hit = self._dimension(d, code, ctx)\n        kap, s, m, beta = hit[0], hit[2], hit[6], hit[7]\n        T, y = self.T, self.opindex[op]\n        nm, yv = ~m, self.Y[:T] == y\n        Z = float(kap.sum())\n        p0y = (float(kap[yv].sum()) + A * self._base(op)) / (Z + A)\n        dp0 = -beta * (float((nm & yv).sum()) - p0y * float(nm.sum())) / (Z + A)\n        sm = self.sumL[d - 1].get(code)\n        P, dlog, dE = np.zeros(len(picked)), np.zeros(len(picked)), np.zeros(len(picked))\n        for j, k in enumerate(picked):\n            dE[j] = -beta * (self.tot[k] - (sm[k] if sm is not None else 0.0))\n            f = self.keys[k]\n            if f is None:\n                P[j], dlog[j] = p0y, dp0 / p0y\n            elif f[0] == "num":\n                pred = self.numeric.predict(ctx.get(f))\n                P[j] = p0y if pred is None else 0.01 * p0y + 0.99 * (pred == op)\n                dlog[j] = (1.0 if pred is None else 0.01) * dp0 / P[j]\n            else:\n                mm = self.V[:T, k] == self._code(ctx.get(f))\n                C, N = float(kap[mm & yv].sum()), float(kap[mm].sum())\n                P[j] = (C + A * p0y) / (N + A)\n                dlog[j] = ((-beta * float((mm & yv & nm).sum()) + A * dp0) / (C + A * p0y)\n                           + beta * float((mm & nm).sum()) / (N + A))\n        r = a * P / max(float(a @ P), 1e-300)\n        return float((r - a) @ dE + r @ dlog)\n\n    def observe(self, ctx, op, when=0):\n        """Each head\'s and each key\'s prequential log probability of the operator, each head\'s theta step, then the\n        episode is stored."""\n        if not self.conditional:\n            self.n0[op] += 1\n            self.n += 1\n            return\n        self._ensure(ctx, op)\n        T, y = self.T, self.opindex[op]\n        live = np.array([k for k, f in enumerate(self.keys) if self._readable(f, ctx)])\n        vals = np.array([self._code(ctx.get(self.keys[k]), True) if k else -1 for k in live], np.int32)\n        q = self._situation(ctx, True)\n        lw = self._weights()\n        pd = np.zeros(len(lw))\n        best = (int(np.argmax(lw)), None)\n        for d in range(len(lw)):                        # each head\'s probability of what happened\n            p, pr, picked, w, _ = self._dimension(d, int(q[d - 1]) if d else 0, ctx)\n            pd[d] = max(p.get(op, pr), 1e-300)\n            if d == best[0]:\n                best = (d, dict(zip(picked, w)))\n        if T:                                           # each head\'s similarity weight: one gradient step\n            g = np.array([self._grad(d, int(q[d - 1]), ctx, op) for d in range(1, len(lw))])\n            self.rms = 0.9 * self.rms + 0.1 * g ** 2\n            self.theta = np.clip(self.theta + ETA * g / (np.sqrt(self.rms) + 1e-8), 0.0, 20.0)\n        dims = np.argsort(-lw)[:DTOP]\n        w = np.exp(lw[dims] - lw[dims].max())\n        pmix = float((w / w.sum()) @ pd[dims])          # the column\'s probability of what happened\n        self.ev = self.ev + np.log(pd)\n        yv = self.Y[:T] == y                            # each key\'s prequential log probability, counting every episode\n        B = self.V[:T, live] == vals[None, :]\n        p0y = (yv.sum() + A * self._base(op)) / (T + A)\n        P = ((yv[:, None] & B).sum(0) + A * p0y) / (B.sum(0) + A)\n        P[0] = p0y\n        for j, k in enumerate(live):                    # the numeric expert is scored by its formula\n            f = self.keys[k]\n            if f is not None and f[0] == "num":\n                pred = self.numeric.predict(ctx.get(f))\n                P[j] = p0y if pred is None else 0.01 * p0y + 0.99 * (pred == op)\n        self.S[T] = q\n        self.V[T] = -1\n        self.V[T, live] = vals\n        self.L[T] = math.log(max(p0y, 1e-300))          # an absent key backs off\n        self.L[T, live] = np.log(np.maximum(P, 1e-300))\n        self.Y[T], self.when[T] = y, when\n        self._account(T, +1)\n        self.T += 1\n        if self.T >= MEMORY:                            # forget the oldest quarter\n            d = MEMORY // 4\n            for t in range(d):\n                self._account(t, -1)\n            for arr in (self.S, self.V, self.L, self.Y, self.when):\n                arr[:self.T - d] = arr[d:self.T]\n            self.T -= d\n        self.cache = {}\n        self.last = {self.keys[k]: float(wk) for k, wk in best[1].items()}\n        self.last_dim = None if best[0] == 0 else self.sfeats[best[0] - 1]\n        self.n0[op] += 1\n        if ("num",) in ctx and isinstance(op, tuple) and len(op) == 3:\n            self.numeric.add(ctx[("num",)], op, missed=pmix < 0.5)\n        self.n += 1\n\n    def _account(self, t, sign):\n        """Add (or remove) episode t\'s log probabilities to the running sums."""\n        self.tot = self.tot + sign * self.L[t]\n        for i, sl in enumerate(self.sumL):\n            c = int(self.S[t, i])\n            sl[c] = sl.get(c, np.zeros(len(self.keys))) + sign * self.L[t]\n\n    def restate(self, f, value_at):\n        """Rewrite the stored value of situation feature f (a hidden-state column relabelled after a fit): value_at maps the\n        time an episode was seen to the feature\'s value then."""\n        i = self.sindex.get(f)\n        if i is None:\n            return\n        for t in range(self.T):\n            v = value_at(int(self.when[t]))\n            if v is not None:\n                self.S[t, i] = self._code(v, True)\n        sl = {}\n        for t in range(self.T):\n            c = int(self.S[t, i])\n            sl[c] = sl.get(c, np.zeros(len(self.keys))) + self.L[t]\n        self.sumL[i] = sl\n        self.cache = {}\n\n    def attended(self):\n        return max(self.last, key=self.last.get) if self.last else None\n\n    def situation(self):\n        """The most weighted head\'s situation feature (None: no situation)."""\n        return self.last_dim\n\n    def trusted(self):\n        """The key with the most evidence over every episode (for the hidden-state area\'s evidence)."""\n        if not self.T:\n            return None\n        return self.keys[int(np.argmax(np.array(self.prior) + self.tot))]\n\n\nclass Column:\n    def __init__(self, cid, frame, state, born, conditional=True, kind_prior=None, library=None):\n        self.cid, self.frame, self.state, self.born = cid, frame, state, born\n        self.conditional, self.kind_prior, self.library = conditional, kind_prior, library\n        self.unit = 0                                   # the gcd of the moves seen: the lattice quotients are taken in\n        self.content = Spotlight()                      # the template, addressed by cell index in the frame\n        for i, c in enumerate(getattr(frame, "colours", [])):\n            self.content.write(i, "colour", int(c))\n        self.keys = {}                                  # akey -> ConditionalOps (the action driver)\n        self.clock = ConditionalOps(frame.K, frame.identity, 0.5, conditional=False)\n        self.ev = {"action": 0.0, "clock": 0.0}\n        self.active, self.uses = True, 0\n        self.reach = 4                                  # the context window\'s margin: grows with the largest move seen\n\n    @property\n    def kind(self):\n        """What carries across levels: the frame type and the content\'s shape and colours."""\n        f = self.frame\n        c = f.cells0 - f.cells0.min(0)\n        return (f.kind, tuple(map(tuple, c.tolist())), tuple(int(v) for v in getattr(f, "colours", [])))\n\n    def akey(self, action, state=None):\n        aid, x, y = action\n        if aid != 6:\n            return ("key", aid)\n        s = self.state if state is None else state\n        if getattr(self, "_cellset", (None,))[0] != s:  # cached per state: the click test runs inside the search\n            self._cellset = (s, set(map(tuple, self.frame.place(s).tolist())))\n        return ("self",) if (x, y) in self._cellset[1] else ("at", x, y)\n\n    def belief(self, akey):\n        if akey not in self.keys:\n            self.keys[akey] = ConditionalOps(self.frame.K, self.frame.identity, P_IDENTITY[akey[0]],\n                                             conditional=self.conditional and akey[0] != "at",\n                                             kind_prior=self.kind_prior, library=self.library)\n        return self.keys[akey]\n\n    def driver(self):\n        return "clock" if self.ev["clock"] > self.ev["action"] else "action"\n\n    def dist(self, akey, ctx):\n        b = self.clock if self.driver() == "clock" else self.belief(akey)\n        return b.dist({} if b is self.clock else ctx)\n\n    def learn(self, akey, ctx, op, when=0):\n        """Prequential evidence for each driver, then both learn the observed operator (when: the brain\'s step)."""\n        b = self.belief(akey)\n        lp_a = math.log(max(b.predictive(ctx, op), 1e-12))\n        lp_c = math.log(max(self.clock.predictive({}, op), 1e-12))\n        self.last_lp = lp_c if self.driver() == "clock" else lp_a   # this step\'s evidence, for the PFC (part 24)\n        self.ev["action"] += lp_a\n        self.ev["clock"] += lp_c\n        b.observe(ctx, op, when)\n        self.clock.observe({}, op)\n        self.uses += 1\n        if isinstance(op, tuple) and len(op) == 3 and all(isinstance(v, int) for v in op):\n            self.reach = min(16, max(self.reach, abs(op[0]) + 1, abs(op[1]) + 1))   # local context only\n            for v in op[:2]:\n                if v:\n                    self.unit = math.gcd(self.unit, abs(v))\n\n    def harvest(self):\n        """What this column learned that carries to other games: for each key, the kind of its most-attended expert,\n        relative to the key\'s usual operator."""\n        out = []\n        for akey, b in self.keys.items():\n            if akey[0] != "key" or b.n < 3:\n                continue\n            op = b.n0.most_common(1)[0][0]\n            out.append(feature_kind(b.attended(), op))\n        return out\n', 'executive.py': '"""The executive (ARCHITECTURE.md section 6), a stand-in for the PFC: goal hypotheses, search over the brain\'s model,\nand the choice of plan. Part 15b\'s lexicographic satisficing agent with part 17\'s commitment.\n\nGoal hypotheses are about kinds (a column\'s content signature), so evidence carries across levels:\n- ("on", kind, c): some column of this kind lies wholly on scene colour c;\n- ("at", kind, state): some column of this kind is in this state;\n- ("joint", states): the columns are in this joint state, a conjunction over columns (part 21: a level\'s WIN usually\n  needs several columns in particular states at once, so visiting one column\'s state with the others wrong must not\n  refute it).\nPrior (part 22): each hypothesis has a type that carries across games ("on-rare": an "on" whose colour covers at most\nRARE x the column\'s size on the screen, such as a marker, exit or slot; "on-common"; "at"; "joint"). A hypothesis\'s\nprior is P(type) / (the number of hypotheses of that type); "at" is spread over AT_STATES states per kind and "joint"\nover JOINT_STATES. P(type) is an overhypothesis (overhypothesis.py): at every WIN, the type of the most probable\nunrefuted hypothesis that held, scored with even type priors, is one draw. Warm-up play on other games supplies the\nfirst draws.\n\nThe choice of plan (part 22, part 17\'s DE17): the plan that leaves the least drive D = (dp^3 + du^3 + de^3)^(1/2):\ndp = (1 - P(WIN along it)) x hunger (0 once P(WIN) >= ALPHA), hunger = (actions since the level started or the last\nWIN + 30) / 25; du = the share of the reachable information it leaves unlearned; de = its length / 50. A hypothesis that holds without a WIN gains an error; at a WIN, every hypothesis that\ndidn\'t hold gains one. Hypotheses first registered after WINs start charged with them (part 17).\n"""\nimport math\nimport time\nfrom collections import deque\n\nimport numpy as np\n\nfrom arc_agent.frames import H, W\nfrom arc_agent.overhypothesis import Overhypothesis\n\nALPHA, THETA, STEP_BITS = 0.5, 0.05, 0.02\nDEPTH, MAX_NODES, AT_STATES, JOINT_STATES = 60, 3000, 4096, 10 ** 6   # the node cap bounds the search\nBUDGET = 0.3                                            # seconds per search: an anytime search, the best so far\nLOG_EPS = math.log(0.01)\nTYPES, RARE, EFFORT = ("on-rare", "on-common", "at", "joint"), 4, 50\nFIXED_HUNGER = None                                     # ablation: a constant hunger instead of a growing one\n\n\ndef h2(p):\n    return 0.0 if p <= 0 or p >= 1 else -p * math.log2(p) - (1 - p) * math.log2(1 - p)\n\n\nclass Goals:\n    def __init__(self, type_draws=()):\n        self.over = Overhypothesis()                    # the prior over goal types (part 22)\n        for t in type_draws:                            # warm-up: WIN types from other games\n            self.over.add("goal", t)\n        self.win_types = []                             # this game\'s draws\n        self.on_type = {}                               # (kind, colour) -> "on-rare" | "on-common"\n        self.on = {}                                    # (kind, colour) -> errors\n        self.at = {}                                    # kind -> {state: errors}\n        self.at_rest = {}                               # kind -> errors shared by every state never named\n        self.joint, self.joint_rest = {}, 0             # joint state -> errors; errors of every joint state never named\n        self.wins = 0\n\n    def register(self, kinds, colours, static=None):\n        if static is not None:                          # an "on" is rare where its colour covers little of the screen\n            counts = dict(zip(*np.unique(static, return_counts=True)))\n            for k in kinds:\n                for c in colours:\n                    self.on_type[(k, c)] = "on-rare" if counts.get(c, 0) <= RARE * len(k[1]) else "on-common"\n        for k in kinds:\n            if k not in self.at:\n                self.at[k], self.at_rest[k] = {}, self.wins\n            for c in colours:\n                if (k, c) not in self.on:\n                    self.on[(k, c)] = self.wins\n\n    def type_prior(self, even=False):\n        return {t: 1 / len(TYPES) for t in TYPES} if even else self.over.prior("goal", TYPES)\n\n    def _weights(self, even=False):\n        P = self.type_prior(even)\n        n_type = {t: max(1, sum(1 for key in self.on if self.on_type.get(key, "on-common") == t)) for t in TYPES[:2]}\n        n_k = max(len(self.at), 1)\n        w_on = {key: math.log(P[self.on_type.get(key, "on-common")] / n_type[self.on_type.get(key, "on-common")])\n                + LOG_EPS * e for key, e in self.on.items()}\n        base = {k: math.log(P["at"] / n_k / AT_STATES) for k in self.at}\n        self._bj = math.log(P["joint"] / JOINT_STATES)\n        return w_on, base\n\n    def mass(self, held):\n        """Posterior probability that the WIN is caused by one of the hypotheses in `held`."""\n        w_on, base = self._weights()\n        bj = self._bj\n        m = max(list(w_on.values()) + [b + LOG_EPS * self.at_rest[k] for k, b in base.items()] +\n                [bj + LOG_EPS * self.joint_rest])\n        z, num = 0.0, 0.0\n        z += (JOINT_STATES - len(self.joint)) * math.exp(bj + LOG_EPS * self.joint_rest - m)\n        for s, e in self.joint.items():\n            z += math.exp(bj + LOG_EPS * e - m)\n        for h in held:\n            if h[0] == "joint":\n                num += math.exp(bj + LOG_EPS * self.joint.get(h[1], self.joint_rest) - m)\n        for key, w in w_on.items():\n            v = math.exp(w - m)\n            z += v\n            if ("on",) + key in held:\n                num += v\n        for k, b in base.items():\n            named = self.at[k]\n            z += (AT_STATES - len(named)) * math.exp(b + LOG_EPS * self.at_rest[k] - m)\n            for s, e in named.items():\n                v = math.exp(b + LOG_EPS * e - m)\n                z += v\n            for h in held:\n                if h[0] == "at" and h[1] == k:\n                    e = named.get(h[2], self.at_rest[k])\n                    num += math.exp(b + LOG_EPS * e - m)\n        return num / z if z > 0 else 0.0\n\n    def no_win(self, held):\n        for h in held:\n            if h[0] == "on" and (h[1], h[2]) in self.on:\n                self.on[(h[1], h[2])] += 1\n            elif h[0] == "at" and h[1] in self.at:\n                self.at[h[1]][h[2]] = self.at[h[1]].get(h[2], self.at_rest[h[1]]) + 1\n            elif h[0] == "joint":\n                self.joint[h[1]] = self.joint.get(h[1], self.joint_rest) + 1\n\n    def win_type(self, held):\n        """The type of the most probable unrefuted hypothesis that held at the WIN, scored with even type priors."""\n        w_on, base = self._weights(even=True)\n        best = None\n        for h in held:\n            if h[0] == "on" and (h[1], h[2]) in w_on:\n                w, t = w_on[(h[1], h[2])], self.on_type.get((h[1], h[2]), "on-common")\n            elif h[0] == "at" and h[1] in base:\n                w, t = base[h[1]] + LOG_EPS * self.at[h[1]].get(h[2], self.at_rest[h[1]]), "at"\n            elif h[0] == "joint":\n                w, t = self._bj + LOG_EPS * self.joint.get(h[1], self.joint_rest), "joint"\n            else:\n                continue\n            if best is None or w > best[0]:\n                best = (w, t)\n        return None if best is None else best[1]\n\n    def win(self, held):\n        t = self.win_type(held)\n        if t is not None:                               # one draw for the overhypothesis over goal types\n            self.over.add("goal", t)\n            self.win_types.append(t)\n        for key in self.on:\n            if ("on",) + key not in held:\n                self.on[key] += 1\n        for k in self.at:\n            keep = {h[2] for h in held if h[0] == "at" and h[1] == k}\n            for s in list(self.at[k]):\n                if s not in keep:\n                    self.at[k][s] += 1\n            for s in keep:\n                self.at[k].setdefault(s, self.at_rest[k])\n            self.at_rest[k] += 1\n        keep = {h[1] for h in held if h[0] == "joint"}\n        for s in list(self.joint):\n            if s not in keep:\n                self.joint[s] += 1\n        for s in keep:\n            self.joint.setdefault(s, self.joint_rest)\n        self.joint_rest += 1\n        self.wins += 1\n\n\nclass Executive:\n    def __init__(self, brain, type_draws=()):\n        self.brain = brain\n        self.goals = Goals(type_draws)\n        self.since = 0                                  # actions since the level started or the last WIN\n        self.plan = []                                  # [(action, expected joint state after it)]\n        self.kind_of_plan = None\n        self.start_grid = None                          # the level\'s first screen: where RESET returns\n        self.can_reset = False\n\n    # ---- what holds in a joint state -----------------------------------------------------------------------------\n    def held(self, states, static):\n        out = {("joint", tuple(sorted(states.items())))}\n        cols = {c.cid: c for c in self.brain.planning()}\n        for cid, s in states.items():\n            c = cols[cid]\n            k = c.kind\n            out.add(("at", k, s))\n            if c.frame.kind == "translation":\n                p = c.frame.place(s)\n                if np.all((p[:, 0] >= 0) & (p[:, 0] < W) & (p[:, 1] >= 0) & (p[:, 1] < H)):\n                    under = set(int(v) for v in static[p[:, 1], p[:, 0]])\n                    if len(under) == 1:\n                        out.add(("on", k, under.pop()))\n        return out\n\n    def register(self, static):\n        kinds = [c.kind for c in self.brain.planning() if c.frame.kind == "translation"]\n        colours = [int(v) for v in np.unique(static)]\n        self.goals.register(kinds, colours, static)\n        for c in self.brain.planning():\n            if c.kind not in self.goals.at:\n                self.goals.register([c.kind], [])\n\n    def observe(self, win_states=None):\n        """After a step without a WIN: what holds now is refuted. At a WIN: `win_states` is the predicted joint state\n        the last action led to."""\n        static = self.brain.static_scene()\n        self.register(static)\n        if win_states is not None:\n            self.goals.win(self.held(win_states, static))\n        else:\n            self.goals.no_win(self.held(dict(self.brain.joint()), static))\n\n    # ---- search over the model --------------------------------------------------------------------------------\n    def actions(self, available):\n        keys = [(a, None, None) for a in available if a not in (0, 6)]\n        clicks = []\n        if 6 in available:\n            seen = set()\n            for c in self.brain.planning():             # a click on each column, wherever it is\n                x, y = c.frame.place(c.state)[0].tolist()\n                if 0 <= x < W and 0 <= y < H:\n                    clicks.append((6, int(x), int(y), ("click-col", c.cid)))\n                    seen.add((int(x), int(y)))\n            for x, y, kind in self.brain.m1.proposals(self.brain.v1.scene, self.brain.v1.ground, self.brain.claimed()):\n                if (x, y) not in seen:\n                    clicks.append((6, x, y, kind))\n        return keys, clicks\n\n    def reset_states(self):\n        """Where each planning column is on the level\'s first screen (RESET\'s known effect)."""\n        out = {}\n        for c in self.brain.planning():\n            s = c.state\n            if c.frame.kind == "translation" and self.start_grid is not None:\n                hits = c.frame.find_all(self.start_grid)\n                s = min(hits, key=lambda h: abs(h[0]) + abs(h[1])) if hits else s\n            elif c.frame.kind == "recolour" and self.start_grid is not None:\n                s = c.frame.read(self.start_grid)\n            out[c.cid] = s\n        return out\n\n    def search(self, available):\n        brain = self.brain\n        brain.cache.clear()                             # the model has changed since the last search\n        static = brain.static_scene()\n        start = dict(brain.joint())\n        keys, clicks = self.actions(available)\n        root = [(a, ("key", a[0])) for a in keys] + [((6, x, y), k) for _, x, y, k in clicks]\n        deep = [(a, k) for a, k in root if k[0] == "key" or k[0] == "click-col" or brain.m1.effect.get(k, (0, 0))[0] > 0]\n        if self.can_reset and self.start_grid is not None:   # RESET: a known effect, so plans may use it\n            restart = self.reset_states()\n            root, deep = root + [((0, None, None), ("reset",))], deep + [((0, None, None), ("reset",))]\n        held0 = self.held(start, static)\n        out = []                                        # (path, pwin, info); a path is [(action, expected state)]\n        q = deque([(start, [], held0, 0.0, 0.0)])\n        seen = {tuple(sorted(start.items()))}\n        n, t0 = 0, time.time()\n        while q and n < MAX_NODES and time.time() - t0 < BUDGET:\n            states, path, held, pwin, info = q.popleft()\n            n += 1\n            if len(path) >= DEPTH:\n                continue\n            for action, kind in (root if not path else deep):\n                if action[0] == 0:\n                    new, bits = {cid: restart.get(cid, s) for cid, s in states.items()}, 0.0\n                else:\n                    new, bits = brain.model_step(states, action, static)\n                if new == states and bits == 0.0 and not path and kind[0] not in ("click-col", "reset") and not brain.m1.tried(action) and not brain.modelled(action):\n                    bits = h2(brain.m1.p_effect(kind))  # no column has a model of it, and never tried here\n                if bits >= THETA:                       # an uncertain outcome: a leaf worth its bits\n                    out.append((path + [(action, None)], pwin, info + bits))\n                    continue\n                if new == states:\n                    continue\n                key = tuple(sorted(new.items()))\n                if key in seen:\n                    continue\n                seen.add(key)\n                h = self.held(new, static)\n                fresh = h - held\n                m = self.goals.mass(fresh) if fresh else 0.0\n                node = (new, path + [(action, key)], held | h, pwin + (1 - pwin) * m, info + h2(m))\n                out.append((node[1], node[3], node[4]))\n                q.append(node)\n        return out\n\n    def decide(self, available):\n        """(action, expected joint state after it), from the committed plan or a new one; None if there is no plan\n        of any kind."""\n        if self.plan:\n            return self.plan.pop(0)\n        cands = self.search(available)\n        if not cands:\n            return None\n        hunger = FIXED_HUNGER if FIXED_HUNGER is not None else (self.since + 30) / 25\n        u0 = max(c[2] for c in cands)\n\n        def drive(c):                                   # part 17\'s DE17: deviations weighted by urgency\n            path, pwin, info = c\n            dp = 0.0 if pwin >= ALPHA else (1 - pwin) * hunger\n            du = 0.0 if u0 < THETA else max(0.0, u0 - info - THETA) / u0\n            de = len(path) / EFFORT\n            return (dp ** 3 + du ** 3 + de ** 3) ** 0.5\n\n        best = min(cands, key=lambda c: (round(drive(c), 12), len(c[0])))\n        if best[1] <= 0 and best[2] < THETA:\n            return None\n        path = best[0]\n        kind = "satisfice" if best[1] >= ALPHA else "explore" if best[2] >= THETA else "approach"\n        self.plan, self.kind_of_plan = list(path), kind\n        chosen = next(c for c in cands if c[0] is path)\n        self.last_choice = dict(kind=kind, length=len(path), pwin=round(chosen[1], 4), info=round(chosen[2], 3),\n                                candidates=len(cands), leaf=path[-1][1] is None)\n        return self.plan.pop(0)\n\n    def fallback(self, available):\n        """No plan of any kind: the action whose outcome is most uncertain here, else an untried click, else None."""\n        brain = self.brain\n        static = brain.static_scene()\n        start = dict(brain.joint())\n        keys, clicks = self.actions(available)\n        best = None\n        for a in keys + [(6, x, y) for _, x, y, _ in clicks]:\n            _, bits = brain.model_step(start, a, static)\n            if a[0] == 6 and not brain.m1.tried(a):\n                bits = max(bits, 1e-3)\n            if bits > 0 and (best is None or bits > best[0]):\n                best = (bits, a)\n        return None if best is None else (best[1], None)\n\n    def surprise(self, expected):\n        """Drop the plan if the brain is not where the plan expected it to be."""\n        if expected is not None and tuple(sorted(dict(self.brain.joint()).items())) != expected:\n            self.plan = []\n', 'frames.py': '"""Reference frames for assoc columns (ARCHITECTURE.md sections 2-3): each is a coordinate system with a group of\noperators, and lays its column\'s content out on the screen.\n\n- translation: a rigid set of cells (part 12: translations in the world frame, quarter turns about its own pivot in\n  its body frame). State: (tx, ty, r). Operator: (dx, dy, k); (0, 0, 0) is the identity.\n- recolour: fixed cells whose colours change together by one colour map. State: the colours, in cell order.\n  Operator: a colour map, as a sorted tuple of (old, new) pairs; () is the identity.\n- bar: an ordered line of cells filled one at a time (Z_n, a count). State: how many cells are filled. Operator: a\n  step (+1, -1, 0). The counters and timers the games draw are bars or translations driven by the clock.\n\nStructure (the offsets, the order) is kept apart from content (the colours), so content can be relabelled without\ntouching structure.\n"""\nimport numpy as np\n\n\nH = W = 64\n\n\ndef rot(p, r):\n    """Part 12-13\'s 90-degree rotations of the plane: points p (n x 2) turned r quarter turns."""\n    p = np.array(p).reshape(-1, 2)\n    for _ in range(r % 4):\n        p = np.stack([-p[:, 1], p[:, 0]], -1)\n    return p\n\n\nclass Translation:\n    """A rigid set of cells (part 12): translations act in the world frame, 90-degree rotations about the piece\'s own\n    pivot act in its body frame. State (tx, ty, r): the cells are rot(cells0 - pivot, r) + pivot + (tx, ty).\n    Operator (dx, dy, k): move the pivot by (dx, dy) and turn the piece by k quarter turns about it."""\n    kind = "translation"\n    identity = (0, 0, 0)\n    K = 33 * 33 * 4                                     # displacements within 16 cells, times 4 turns\n\n    def __init__(self, cells, colours):\n        self.cells0 = np.array(cells, int)              # (n, 2) as (x, y), where the column was found\n        self.colours = np.array(colours, np.int8)\n        self.pivot = np.round(self.cells0.mean(0)).astype(int)\n        self.rel = {r: rot(self.cells0 - self.pivot, r).astype(int) for r in range(4)}\n\n    def place(self, state):\n        return self.rel[state[2] % 4] + self.pivot + np.array(state[:2], int)\n\n    def apply(self, state, op):\n        return (state[0] + op[0], state[1] + op[1], (state[2] + op[2]) % 4)\n\n    def inside(self, state):\n        p = self.place(state)\n        return bool(np.all((p[:, 0] >= 0) & (p[:, 0] < W) & (p[:, 1] >= 0) & (p[:, 1] < H)))\n\n    def draw(self, grid, state):\n        p = self.place(state)\n        ok = (p[:, 0] >= 0) & (p[:, 0] < W) & (p[:, 1] >= 0) & (p[:, 1] < H)\n        grid[p[ok, 1], p[ok, 0]] = self.colours[ok]\n\n    def matches(self, grid, state):\n        if not self.inside(state):\n            return False\n        p = self.place(state)\n        return bool(np.all(grid[p[:, 1], p[:, 0]] == self.colours))\n\n    def score(self, grid, state):\n        """The fraction of the column\'s cells whose colour is seen at this state (0 if any cell is off the screen)."""\n        if not self.inside(state):\n            return 0.0\n        p = self.place(state)\n        return float(np.mean(grid[p[:, 1], p[:, 0]] == self.colours))\n\n    def _rings(self, near, radius, turns):\n        for r in range(radius + 1):                     # nearest first\n            for dx in range(-r, r + 1):\n                for dy in (-r, r) if abs(dx) != r else range(-r, r + 1):\n                    for k in turns:\n                        yield (near[0] + dx, near[1] + dy, (near[2] + k) % 4)\n\n    def find(self, grid, near, radius=16):\n        """The state nearest `near` at which the content is seen exactly: same turn first, then the other turns."""\n        for s in self._rings(near, radius, (0,)):\n            if self.matches(grid, s):\n                return s\n        for s in self._rings(near, 4, (1, 2, 3)):\n            if self.matches(grid, s):\n                return s\n        return None\n\n    def find_near(self, grid, near, radius, frac):\n        """The best-matching state within radius (at least frac of the cells seen), nearest on ties, or None."""\n        best = None\n        for i, s in enumerate(self._rings(near, radius, (0, 1, 2, 3))):\n            sc = self.score(grid, s)\n            if sc >= frac and (best is None or sc > best[0]):\n                best = (sc, s)\n        return None if best is None else best[1]\n\n    def find_all(self, grid):\n        """Every state at which the content is seen exactly (for a new level)."""\n        out = []\n        for r in range(4):\n            x0, y0 = self.rel[r][0] + self.pivot\n            ys, xs = np.nonzero(grid == self.colours[0])\n            for x, y in zip(xs, ys):\n                s = (int(x - x0), int(y - y0), r)\n                if self.matches(grid, s):\n                    out.append(s)\n        return out\n\n    def op_between(self, s0, s1):\n        return (s1[0] - s0[0], s1[1] - s0[1], (s1[2] - s0[2]) % 4)\n\n\nclass Choice:\n    """A categorical frame (part 24, the PFC column\'s): which of a set of options holds. State: an option, or None\n    before any. Operator: ("to", option), or ("stay",), the identity."""\n    kind = "choice"\n    identity = ("stay",)\n\n    def __init__(self, options):\n        self.options = tuple(options)\n        self.K = len(self.options) + 1\n        self.cells0 = np.zeros((1, 2), int)\n\n    def apply(self, state, op):\n        return state if op == ("stay",) else op[1]\n\n    def op_between(self, s0, s1):\n        return ("stay",) if s0 == s1 else ("to", s1)\n\n\nclass Recolour:\n    kind = "recolour"\n    identity = ()\n    K = 256\n\n    def __init__(self, cells, colours):\n        self.cells0 = np.array(cells, int)\n        self.colours = np.array(colours, np.int8)       # the colours when found (the first state)\n\n    def place(self, state=None):\n        return self.cells0\n\n    def apply(self, state, op):\n        m = dict(op)\n        return tuple(m.get(c, c) for c in state)\n\n    def inside(self, state):\n        return True\n\n    def draw(self, grid, state):\n        grid[self.cells0[:, 1], self.cells0[:, 0]] = np.array(state, np.int8)\n\n    def read(self, grid):\n        return tuple(int(v) for v in grid[self.cells0[:, 1], self.cells0[:, 0]])\n\n    def op_between(self, s0, s1):\n        m = {}\n        for a, b in zip(s0, s1):\n            if a != b:\n                if m.get(a, b) != b:\n                    return None                         # not one colour map: the frame fails here\n                m[a] = b\n        return tuple(sorted(m.items()))\n\n\nclass Bar:\n    kind = "bar"\n    identity = 0\n    K = 3\n\n    def __init__(self, cells, empty, full):\n        self.cells0 = np.array(cells, int)              # in fill order\n        self.empty, self.full = int(empty), int(full)\n\n    def place(self, state=None):\n        return self.cells0\n\n    def apply(self, state, op):\n        return max(0, min(len(self.cells0), state + op))\n\n    def inside(self, state):\n        return True\n\n    def draw(self, grid, state):\n        c = self.cells0\n        grid[c[:state, 1], c[:state, 0]] = self.full\n        grid[c[state:, 1], c[state:, 0]] = self.empty\n\n    def read(self, grid):\n        v = grid[self.cells0[:, 1], self.cells0[:, 0]]\n        return int(np.sum(v == self.full))\n\n    def op_between(self, s0, s1):\n        return int(np.sign(s1 - s0))\n\n    def extend(self, cell):\n        self.cells0 = np.vstack([self.cells0, np.array(cell, int)[None]])\n', 'hidden.py': '"""The hidden-state area (docs/HIDDEN_STATE.md sections 7-8; docs/AGENT.md part 27): columns whose state is not on the screen.\n\n- Delay columns: the event 1, 2, ..., LAGS steps ago. Each is an ordinary Column over a Choice frame, and what an event\n  does to it is learned like any column\'s (lag k copies lag k - 1): nothing but the wiring makes it a delay line.\n  Similar columns specialise by lag, as time cells do by delay.\n- A graph column: an ordinary Column over a Choice frame of KG states whose state is inferred, not read. An HMM with\n  event-conditioned transitions, fitted by Baum-Welch on the stored history (cut into CHUNK-step pieces), whose\n  evidence is what the visible columns saw: read from their own memories at fit time (no second copy), each episode\'s\n  operator given the value of that column\'s most trusted key now. Starts keep their state (a state persists until an\n  event changes it). After each fit the labels are matched to the old ones (the assignment that agrees most) and every\n  stored episode\'s graph state is rewritten, wherever it is read.\n\nTheir states reach the visible columns as ordinary situation features ("col", cid): Brain.others().\n"""\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\n\nfrom arc_agent.column import Column\nfrom arc_agent.frames import Choice\n\nLAGS, KG = 4, 6\nFIT_EVERY, CHUNK, STARTS, ITERS, STAY, PT, PE = 100, 50, 8, 40, 0.8, 0.01, 0.1\nHISTORY = 3000                                          # steps kept for fitting\n\n\nclass Graph:\n    """The HMM: pi (KG), Tr (events + 1, KG, KG; the last is the identity, for padding), Em (observers, values, operators,\n    KG). Sizes grow with the codes; a warm start pads the old parameters."""\n\n    def __init__(self, rng):\n        self.rng, self.params = rng, None\n\n    def random(self, nE, nM, nV, nY):\n        r = self.rng\n        Tr = r.dirichlet(np.ones(KG), size=(nE, KG))\n        Tr = np.concatenate([STAY * np.eye(KG)[None] + (1 - STAY) * Tr, np.eye(KG)[None]], 0)\n        Em = r.dirichlet(np.ones(nY), size=(nM, nV, KG)).transpose(0, 1, 3, 2)\n        return {"pi": 0.5 * np.eye(KG)[0] + 0.5 * r.dirichlet(np.ones(KG)), "Tr": Tr, "Em": Em}\n\n    def padded(self, nE, nM, nV, nY):\n        """The last parameters, grown to the current sizes (new events keep the state; new emissions are uniform)."""\n        p = self.params\n        Tr = np.concatenate([p["Tr"][:-1], np.repeat(np.eye(KG)[None], nE - (len(p["Tr"]) - 1), 0),\n                             np.eye(KG)[None]], 0)\n        Em = np.full((nM, nV, nY, KG), 1.0 / nY)\n        m, v, y = (min(a, b) for a, b in zip(p["Em"].shape[:3], (nM, nV, nY)))\n        Em[:m, :v, :y] = p["Em"][:m, :v, :y]               # (a column\'s evidence key may have changed: sizes shrink)\n        Em /= Em.sum(2, keepdims=True)\n        return {"pi": p["pi"], "Tr": Tr, "Em": Em}\n\n    @staticmethod\n    def emissions(p, L, T, em):\n        b = np.ones((L, T, KG))\n        if len(em):\n            lb = np.zeros((L, T, KG))\n            np.add.at(lb, (em[:, 0], em[:, 1]), np.log(p["Em"][em[:, 2], em[:, 3], em[:, 4]]))\n            b = np.exp(lb)\n        return b\n\n    def estep(self, p, e, em):\n        L, T = e.shape\n        b = self.emissions(p, L, T, em)\n        alpha, c = np.zeros((L, T, KG)), np.zeros((L, T))\n        a = p["pi"][None] * b[:, 0]\n        for t in range(T):\n            if t:\n                a = np.einsum("lk,lkj->lj", alpha[:, t - 1], p["Tr"][e[:, t - 1]]) * b[:, t]\n            c[:, t] = a.sum(1)\n            alpha[:, t] = a / c[:, t:t + 1]\n        beta = np.ones((L, T, KG))\n        cTr = np.zeros_like(p["Tr"])\n        for t in range(T - 2, -1, -1):\n            w = b[:, t + 1] * beta[:, t + 1] / c[:, t + 1:t + 2]\n            beta[:, t] = np.einsum("lkj,lj->lk", p["Tr"][e[:, t]], w)\n            np.add.at(cTr, e[:, t], alpha[:, t][:, :, None] * p["Tr"][e[:, t]] * w[:, None, :])\n        gam = alpha * beta\n        gam /= gam.sum(-1, keepdims=True)\n        cEm = np.zeros_like(p["Em"])\n        if len(em):\n            np.add.at(cEm, (em[:, 2], em[:, 3], em[:, 4]), gam[em[:, 0], em[:, 1]])\n        return float(np.log(c).sum()), gam[:, 0].sum(0), cTr, cEm\n\n    @staticmethod\n    def mstep(cpi, cTr, cEm):\n        Tr = cTr[:-1] + PT\n        Tr = np.concatenate([Tr / Tr.sum(-1, keepdims=True), np.eye(KG)[None]], 0)\n        Em = cEm + PE\n        return {"pi": (cpi + PT) / (cpi + PT).sum(), "Tr": Tr, "Em": Em / Em.sum(2, keepdims=True)}\n\n    def fit(self, e, em, sizes):\n        best = None\n        for s in range(STARTS):\n            p = self.padded(*sizes) if (s == 0 and self.params is not None) else self.random(*sizes)\n            for _ in range(ITERS):\n                ll, cpi, cTr, cEm = self.estep(p, e, em)\n                p = self.mstep(cpi, cTr, cEm)\n            ll = self.estep(p, e, em)[0]\n            if best is None or ll > best[0]:\n                best = (ll, p)\n        return best[1]\n\n\nclass Hidden:\n    def __init__(self, conditional=True, kind_prior=None, seed=0):\n        self.delays = [Column(-10 - k, Choice(range(8)), None, 0, conditional, kind_prior) for k in range(1, LAGS + 1)]\n        self.graph_col = Column(-20, Choice(range(KG)), 0, 0, conditional, kind_prior)\n        self.graph = Graph(np.random.default_rng(seed))\n        self.codes = {"e": {}, "m": {}}\n        self.levels = [[]]                              # per level: (brain step, event code)\n        self.sources = {}                               # observer (cid, akey) -> its ConditionalOps (the evidence)\n        self.key = {}                                   # observer -> the key index its evidence is read with\n        self.belief = np.eye(KG)[0]\n        self.steps = 0\n\n    def columns(self):\n        return self.delays + [self.graph_col]\n\n    def states(self):\n        return {c.cid: c.state for c in self.columns()}\n\n    def code(self, kind, x, add=True):\n        try:\n            hash(x)\n        except TypeError:\n            x = repr(x)\n        d = self.codes[kind]\n        if x not in d and add:\n            d[x] = len(d)\n        return d.get(x)\n\n    def start(self, new_level):\n        """A new level, or a restart: the hidden machine starts again."""\n        if self.levels[-1]:\n            self.levels.append([])\n        p = self.graph.params\n        self.belief = p["pi"].copy() if p is not None else np.eye(KG)[0]\n        for c in self.delays:\n            c.state = None\n        self.graph_col.state = int(self.belief.argmax())\n\n    def emission(self, m, b, i):\n        """Observer m\'s episode i, as (observer, value of its evidence key, operator) codes."""\n        k = self.key.get(m)\n        if k is None:\n            k = self.key[m] = b.index.get(b.trusted(), 0)\n        v = int(b.V[i, k]) + 2 if k < b.V.shape[1] else 0     # +2: the codes for "absent" and "never stored"\n        return self.code("m", m), v, int(b.Y[i])\n\n    def observe(self, t, aid, event, learned):\n        """One step: what the visible columns just learned (learned: [(observer, its ConditionalOps)]) updates the\n        graph\'s belief, the event moves every hidden-state column, and each learns what the event did to it. Returns True\n        after a fit (the graph\'s stored states were rewritten: Brain.restate)."""\n        try:\n            hash(event)\n        except TypeError:\n            event = repr(event)\n        ecode = self.code("e", event)\n        for m, b in learned:\n            self.sources[m] = b\n        em = [self.emission(m, b, b.T - 1) for m, b in learned]\n        p = self.graph.params\n        if p is not None:                               # the belief: this step\'s evidence, then the event\n            for m, v, y in em:\n                if m < p["Em"].shape[0] and v < p["Em"].shape[1] and y < p["Em"].shape[2]:\n                    self.belief = self.belief * p["Em"][m, v, y]\n            self.belief /= self.belief.sum()\n            if ecode < len(p["Tr"]) - 1:\n                self.belief = self.belief @ p["Tr"][ecode]\n        old = self.states()\n        new = {self.delays[0].cid: event}\n        for k in range(1, LAGS):\n            new[self.delays[k].cid] = old[self.delays[k - 1].cid]\n        new[self.graph_col.cid] = int(self.belief.argmax())\n        akey = ("key", aid)\n        for c in self.columns():                        # each learns what the event did to it, like any column\n            ctx = {("self",): old[c.cid], ("event",): event}\n            ctx.update({("col", o): s for o, s in old.items() if o != c.cid})\n            c.learn(akey, ctx, c.frame.op_between(old[c.cid], new[c.cid]), t)\n            c.state = new[c.cid]\n        self.levels[-1].append((t, ecode))\n        self.steps += 1\n        if self.steps % FIT_EVERY == 0:\n            self.fit()\n            return True\n        return False\n\n    # ---- fitting ------------------------------------------------------------------------------------------------\n    def evidence(self):\n        """Every visible column\'s stored episodes as emissions, read with each one\'s most trusted key now:\n        {brain step: [(m, v, y)]}."""\n        self.key = {m: b.index.get(b.trusted(), 0) for m, b in self.sources.items()}\n        out = {}\n        for m, b in self.sources.items():\n            for i in range(b.T):\n                out.setdefault(int(b.when[i]), []).append(self.emission(m, b, i))\n        return out\n\n    def pieces(self, ev):\n        """The stored history cut into CHUNK-step pieces: events (L, T) and emissions [piece, step, m, v, y]."""\n        steps = sum(([s for s in lev] for lev in self.levels), [])[-HISTORY:]\n        chunks, cur, lev_starts = [], [], set()\n        n = 0\n        for lev in self.levels:\n            lev_starts.add(n)\n            n += len(lev)\n        n0 = n - len(steps)\n        for i, s in enumerate(steps):\n            if cur and (len(cur) == CHUNK or (i + n0) in lev_starts):\n                chunks.append(cur)\n                cur = []\n            cur.append(s)\n        if cur:\n            chunks.append(cur)\n        nE = len(self.codes["e"])\n        e = np.full((len(chunks), CHUNK), nE)\n        em = []\n        for li, ch in enumerate(chunks):\n            for ti, (t, ec) in enumerate(ch):\n                e[li, ti] = ec\n                em += [(li, ti, m, v, y) for m, v, y in ev.get(t, [])]\n        return e, np.array(em, int).reshape(-1, 5)\n\n    def sizes(self, ev):\n        allem = [x for ems in ev.values() for x in ems]\n        return (len(self.codes["e"]), max(1, len(self.codes["m"])), max([v for _, v, _ in allem], default=0) + 1,\n                max([y for _, _, y in allem], default=0) + 1)\n\n    def filtered(self, p, ev, levels=None):\n        """The most probable state before each stored step, filtered through each level ({brain step: state}),\n        and the belief after the last step."""\n        out, a = {}, p["pi"].copy()\n        for lev in (levels or self.levels):\n            a = p["pi"].copy()\n            for t, ec in lev:\n                out[t] = int(a.argmax())\n                for m, v, y in ev.get(t, []):\n                    a = a * p["Em"][m, v, y]\n                a /= a.sum()\n                a = a @ p["Tr"][ec]\n        return out, a\n\n    def fit(self):\n        ev = self.evidence()\n        e, em = self.pieces(ev)\n        sizes = self.sizes(ev)\n        p = self.graph.fit(e, em, sizes)\n        if self.graph.params is not None:               # keep the labels: the assignment that agrees most\n            old, new = self.filtered(self.graph.padded(*sizes), ev)[0], self.filtered(p, ev)[0]\n            agree = np.zeros((KG, KG))\n            for t, s in new.items():\n                agree[s, old[t]] += 1\n            rows, cols = linear_sum_assignment(-agree)\n            perm = np.empty(KG, int)\n            perm[cols] = rows                           # new label rows[i] becomes old label cols[i]\n            p = {"pi": p["pi"][perm], "Tr": np.concatenate([p["Tr"][:-1][:, perm][:, :, perm], p["Tr"][-1:]], 0),\n                 "Em": p["Em"][..., perm]}\n        self.graph.params = p\n        self.history_states, _ = self.filtered(p, ev)\n        _, a = self.filtered(p, ev, [self.levels[-1]])   # the current level\'s belief, under the new model\n        self.belief = a\n        self.graph_col.state = int(a.argmax())\n\n    def value_at(self, t):\n        """The graph column\'s state before brain step t, under the current model (None if not stored)."""\n        return getattr(self, "history_states", {}).get(t)\n', 'library.py': '"""Library learning (legacy/docs/CURRICULUM.md, the experiments that built it; a copy for the agent): templates from the formulas of solved tasks, kept greedily while they\nshorten the corpus\'s description (Stitch-style: one template at a time, rewrite, repeat). Polynomial throughout.\n\nA term is a block: a product of factors with parameters. Candidate templates come from two operators:\n    anti-unification   two terms with the same factor structure generalize to a template: inputs become slots; a\n                       parameter whose two values agree within TOL becomes a fixed constant (their mean), otherwise a\n                       free parameter. At most MAX_FACTORS factors.\n    group folding      in one formula, two or more single-input terms with the same structure and parameters within TOL\n                       fold into one group template, sum over a group of inputs of c_v T(v), parameters shared and fixed.\nDescription length (nats) of the corpus given the library (amended 2026-10-26, see docs/CURRICULUM.md; first\nspecified with 1/2 log n per parameter, under which no template ever paid for itself):\n    sum over learned templates of factors * log(number of primitives) + PARAM_COST * fixed constants\n  + sum over term instances of -log P(template) + PARAM_COST * (free parameters of the instance),\nPARAM_COST = log(120): a constant coded to precision 0.1 over a range of 12, as a compression objective counts it\n(Stitch, DreamCoder count each program\'s whole description, so a template saves its body at every use);\nwith each term (or set of terms, for a group) assigned the cheapest template it instantiates, and\nP(template) = (uses + 1) / total over primitives and templates.\n"""\nimport math\n\nimport numpy as np\n\nfrom arc_agent.symreg import N_PARAMS, PRIMITIVES\n\nTOL, MAX_FACTORS = 0.1, 5\nPARAM_COST = __import__("math").log(120.0)\n\n\ndef structure(block):\n    """Canonical structure of a single-binding term: the factor kinds with the slot pattern, factors sorted."""\n    order = sorted(range(len(block.factors)), key=lambda k: (block.factors[k][0], block.bindings[0][block.factors[k][1]]))\n    first = {}\n    slots = []\n    for k in order:\n        v = block.bindings[0][block.factors[k][1]]\n        first.setdefault(v, len(first))\n        slots.append(first[v])\n    kinds = tuple(block.factors[k][0] for k in order)\n    starts = np.cumsum([0] + [N_PARAMS[block.factors[k][0]] for k in range(len(block.factors))])\n    theta = np.concatenate([block.theta[starts[k]:starts[k + 1]] for k in order]) if len(order) else np.zeros(0)\n    return (kinds, tuple(slots)), theta\n\n\nclass Template:\n    def __init__(self, name, struct, theta, fixed, group=False):\n        self.name, self.struct, self.group = name, struct, group\n        self.theta, self.fixed = np.asarray(theta, float), np.asarray(fixed, bool)\n\n    def size(self):\n        return len(self.struct[0]) + int(self.fixed.sum())\n\n    def matches(self, struct, theta):\n        return struct == self.struct and np.all(np.abs(theta[self.fixed] - self.theta[self.fixed]) <= TOL)\n\n    def n_free(self):\n        return int((~self.fixed).sum())\n\n    def spec(self):\n        """The form symreg2.Finder takes."""\n        kinds, slots = self.struct\n        return dict(name=self.name, factors=list(zip(kinds, slots)), slots=len(set(slots)), theta=self.theta,\n                    free=~self.fixed, group=self.group)\n\n    def describe(self):\n        kinds, slots = self.struct\n        out, at = [], 0\n        for kind, s in zip(kinds, slots):\n            k = N_PARAMS[kind]\n            ps = ["%.3g" % self.theta[at + j] if self.fixed[at + j] else "?" for j in range(k)]\n            out.append(f"{kind}(v{s}{\', \' if ps else \'\'}{\', \'.join(ps)})")\n            at += k\n        return ("sum over a group: " if self.group else "") + " * ".join(out)\n\n\nclass Corpus:\n    """Per solved task: its terms (structure, theta, primitive name) and sample size."""\n    def __init__(self):\n        self.tasks = []\n\n    def add(self, finder_record, n):\n        terms = []\n        for b in finder_record["state"].blocks:\n            if getattr(b, "kind", None) in ("affine", "alog"):        # combined-input atoms: task-specific directions\n                terms.append(dict(struct=((b.kind,), (0,)), theta=np.asarray(b.theta, float), prim=b.kind))\n                continue\n            for bind in b.bindings:                                   # a group block contributes one term per input\n                single = type(b)(b.factors, [bind], b.theta, b.free, b.template)\n                st, th = structure(single)\n                prim = "pow*pow" if st[0] == ("pow", "pow") else st[0][0] if len(st[0]) == 1 else "product"\n                terms.append(dict(struct=st, theta=th, prim=prim))\n        self.tasks.append(dict(terms=terms, n=n))\n\n\ndef primitive_cost_free(term):\n    return len(term["theta"])\n\n\ndef assign(corpus, library):\n    """Each term\'s cheapest template; groups first. Returns (uses per name, list of (name, free params, log n))."""\n    uses = {k: 0 for k in list(PRIMITIVES) + ["product"] + [t.name for t in library]}\n    items = []\n    for task in corpus.tasks:\n        covered = set()\n        for t in library:\n            if not t.group:\n                continue\n            idx = [k for k, term in enumerate(task["terms"]) if k not in covered and t.matches(term["struct"], term["theta"])]\n            if len(idx) >= 2:\n                covered.update(idx)\n                uses[t.name] += 1\n                items.append((t.name, t.n_free(), math.log(task["n"])))\n        for k, term in enumerate(task["terms"]):\n            if k in covered:\n                continue\n            best = (term["prim"], primitive_cost_free(term))\n            for t in library:\n                if not t.group and t.matches(term["struct"], term["theta"]) and t.n_free() < best[1]:\n                    best = (t.name, t.n_free())\n            uses[best[0]] = uses.get(best[0], 0) + 1\n            items.append((best[0], best[1], math.log(task["n"])))\n    return uses, items\n\n\ndef description_length(corpus, library):\n    uses, items = assign(corpus, library)\n    for _ in range(2):                                               # P from uses, uses from cheapest assignment\n        total = sum(u + 1 for u in uses.values())\n        lp = {k: math.log((u + 1) / total) for k, u in uses.items()}\n        dl = sum(len(t.struct[0]) * math.log(len(PRIMITIVES)) + PARAM_COST * int(t.fixed.sum()) for t in library)\n        dl += sum(-lp[name] + PARAM_COST * free for name, free, logn in items)\n    return dl, uses\n\n\ndef candidates(corpus, existing):\n    """Anti-unifications of term pairs, and group foldings within formulas, deduplicated."""\n    terms = [term for task in corpus.tasks for term in task["terms"]]\n    seen, out = {(t.struct, t.group, tuple(np.round(np.where(t.fixed, t.theta, np.nan), 2))) for t in existing}, []\n\n    def offer(struct, theta, fixed, group):\n        # margins index a task\'s own constraints, not inputs: they stay primitives, never part of a template\n        if len(struct[0]) > MAX_FACTORS or (not fixed.any() and not group) or \\\n                {"margin", "affine", "alog"} & set(struct[0]):\n            return\n        key = (struct, group, tuple(np.round(np.where(fixed, theta, np.nan), 2)))\n        if key not in seen:\n            seen.add(key)\n            out.append(Template(f"T{len(existing) + len(out)}", struct, theta, fixed, group))\n\n    by_struct = {}\n    for term in terms:\n        if not {"affine", "alog"} & set(term["struct"][0]):\n            by_struct.setdefault(term["struct"], []).append(term)\n    for struct, group in by_struct.items():\n        for a in range(len(group)):\n            for b in range(a + 1, len(group)):\n                ta, tb = group[a]["theta"], group[b]["theta"]\n                fixed = np.abs(ta - tb) <= TOL\n                offer(struct, np.where(fixed, (ta + tb) / 2, ta), fixed, False)\n    for task in corpus.tasks:                                          # relational folding: two-input terms\n        twos = [t for t in task["terms"] if t["struct"][1] == (0, 1)]\n        for a in range(len(twos)):\n            same = [twos[a]] + [s for s in twos[a + 1:] if s["struct"] == twos[a]["struct"]\n                                and np.all(np.abs(s["theta"] - twos[a]["theta"]) <= TOL)]\n            if len(same) >= 2:\n                theta = np.mean([s["theta"] for s in same], 0)\n                offer(twos[a]["struct"], theta, np.ones(len(theta), bool), True)\n    for task in corpus.tasks:\n        singles = [t for t in task["terms"] if len(set(t["struct"][1])) == 1 and\n                   not {"affine", "alog"} & set(t["struct"][0])]\n        for a in range(len(singles)):\n            same = [singles[a]] + [s for s in singles[a + 1:] if s["struct"] == singles[a]["struct"]\n                                   and np.all(np.abs(s["theta"] - singles[a]["theta"]) <= TOL)]\n            if len(same) >= 2:\n                theta = np.mean([s["theta"] for s in same], 0)\n                offer(singles[a]["struct"], theta, np.ones(len(theta), bool), True)\n    return out\n\n\ndef learn(corpus, library=None, max_new=20):\n    """Greedily add the template that most shortens the description, while any does. Returns (library, uses, log)."""\n    library = list(library or [])\n    dl, uses = description_length(corpus, library)\n    log = [dict(added=None, dl=dl)]\n    for _ in range(max_new):\n        best = None\n        for c in candidates(corpus, library):\n            dl_c, _ = description_length(corpus, library + [c])\n            if dl_c < dl - 1e-9 and (best is None or dl_c < best[0]):\n                best = (dl_c, c)\n        if best is None:\n            break\n        dl, tpl = best\n        tpl.name = f"T{len(library)}"\n        library.append(tpl)\n        log.append(dict(added=tpl.describe(), dl=dl))\n    _, uses = description_length(corpus, library)\n    return library, uses, log\n', 'overhypothesis.py': '"""Overhypotheses (docs/AGENT.md, parts 19 and 22): priors learned across games (Kemp, Perfors & Tenenbaum 2007).\nIn the arc_agent it holds the prior over goal types (executive.Goals), learned from WINs within a game and from\nwarm-up play on other games.\n\nA slot is a variable that recurs in every game (what action 2 does; how a one-cell object responds to contact). Each\ngame contributes one confident value per slot, a draw. The prior for a new game is a Dirichlet-multinomial,\nP(v) = (n_v + alpha * b_v) / (N + alpha), with b the flat prior. alpha, how much games vary, is learned: the prediction\nis averaged over a grid of alphas, each weighted by how well it predicted the past draws in order.\n"""\nimport math\nfrom collections import Counter\n\nALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0)\n\n\nclass Overhypothesis:\n    def __init__(self):\n        self.draws = {}                                     # slot -> [values, in the order the games came]\n\n    def add(self, slot, value):\n        self.draws.setdefault(slot, []).append(value)\n\n    def alpha_posterior(self, slot, values):\n        b = 1.0 / len(values)\n        lw = []\n        for a in ALPHAS:\n            n, w = Counter(), 0.0\n            for i, v in enumerate(self.draws.get(slot, [])):\n                w += math.log((n[v] + a * b) / (i + a))\n                n[v] += 1\n            lw.append(w)\n        m = max(lw)\n        p = [math.exp(w - m) for w in lw]\n        z = sum(p)\n        return [x / z for x in p]\n\n    def prior(self, slot, values):\n        """P(v) for a new game, over the slot\'s possible values."""\n        draws, b = self.draws.get(slot, []), 1.0 / len(values)\n        n, N = Counter(draws), len(draws)\n        post = self.alpha_posterior(slot, values)\n        return {v: sum(w * (n[v] + a * b) / (N + a) for w, a in zip(post, ALPHAS)) for v in values}\n', 'pfc.py': '"""The prefrontal cortex (docs/AGENT.md part 24): a column that learns which column to trust.\n\nWhen several assoc columns, each in a different frame, explain the same change, they form a group of competing models.\nThe responsible one has the largest posterior, log P(kind | context) + the sum of its log predictive probabilities\nsince the group formed: a prior times the prequential evidence (Bayesian model selection; MOSAIC\'s responsibilities).\n\nThe prior is a column\'s prediction. The PFC column is an ordinary Column (column.py), the same learning as every other\ncolumn, over a choice frame (frames.py): its state is the kind of model that predicted best last; its operator is a\nswitch to a kind, or staying. Each step, for each group, it observes which candidate predicted best and learns that,\nby attention over context experts by track record, from game-agnostic context: the region\'s size, colours and shape,\nthe action, the level. One PFC column serves every group, as a reusable model over many instances of its frame. Its\nexperience carries across regions, levels and games, so the higher column learns the lower competition\'s prior\n(hierarchical Bayes): the right model wins after fewer observations as experience grows. Learning to learn, from\nordinary column learning one level up; nothing here targets it.\n\nOnly the responsible column of a group is planned with (Miller & Cohen: PFC biases which pathway carries the task);\nthe others keep learning, latent, and a switch is an ordinary operator, so a changed rule is followed by evidence.\n"""\nimport math\nimport random\n\nfrom arc_agent.column import Column\nfrom arc_agent.frames import Choice\n\nKINDS = ("translation", "recolour", "bar")\nLOST = math.log(1e-4)                   # a candidate that could not find itself predicted its observation this badly\n\n\nclass PFC:\n    def __init__(self, seed=0):\n        self.col = Column(-1, Choice(KINDS), None, 0)\n        self.groups = {}                                # gid -> dict(members, ev, state, ctx)\n        self.member = {}                                # cid -> gid\n        self.next, self.rng = 0, random.Random(seed)\n\n    # ---- groups of competing models ---------------------------------------------------------------------------------\n    def group(self, cids, kinds, context):\n        gid = self.next\n        self.next += 1\n        self.groups[gid] = dict(members=list(cids), kinds=dict(zip(cids, kinds)), ev={c: 0.0 for c in cids},\n                                state=None, ctx=dict(context), tie={c: self.rng.random() * 1e-6 for c in cids})\n        for c in cids:\n            self.member[c] = gid\n        return gid\n\n    def ctx(self, g, act):\n        out = dict(g["ctx"])\n        out[("act",)] = act\n        out[("self",)] = g["state"]\n        return out\n\n    def prior(self, g, act):\n        """P(the next responsible kind) from the PFC column\'s prediction of its own next state."""\n        p, rest = self.col.dist(("pfc",), self.ctx(g, act))\n        out = {}\n        for k in KINDS:\n            q = p.get(("to", k), rest)\n            if k == g["state"]:\n                q += p.get(("stay",), 0.0)\n            out[k] = q\n        z = sum(out.values()) or 1.0\n        return {k: v / z for k, v in out.items()}\n\n    def responsible(self, cid, act=None):\n        """True if this column may be planned with: it belongs to no group, or it is its group\'s responsible model."""\n        gid = self.member.get(cid)\n        if gid is None:\n            return True\n        g = self.groups[gid]\n        if not hasattr(self, "_cache") or self._cache[0] != (gid, g["state"], tuple(g["ev"].values())):\n            pri = self.prior(g, act or "key")\n            best = max(g["members"], key=lambda c: math.log(max(pri[g["kinds"][c]], 1e-12)) + g["ev"][c] + g["tie"][c])\n            self._cache = ((gid, g["state"], tuple(g["ev"].values())), best)\n        return self._cache[1] == cid\n\n    # ---- learning: the evidence, and the PFC column\'s own observation ---------------------------------------------\n    def observe(self, evidence, act):\n        """After a step: each candidate\'s log predictive probability of the group\'s cells (brain._group_evidence:\n        every rival scored on the same cells) is evidence; the kind that predicted best is the PFC column\'s\n        observation, learned like any column\'s. A candidate that could not predict (lost) gets LOST."""\n        for gid, g in self.groups.items():\n            lps = {}\n            for cid in g["members"]:\n                lps[cid] = evidence.get(cid, LOST)\n                g["ev"][cid] += lps[cid]\n            if all(lp == LOST for lp in lps.values()):\n                continue                                # the region is gone: nothing to learn here\n            top = max(lps.values())\n            best = {g["kinds"][c] for c, lp in lps.items() if lp >= top - 1e-9}\n            if len(best) != 1:                          # every candidate predicted as well: no information\n                continue\n            kind = best.pop()\n            op = self.col.frame.op_between(g["state"], kind)\n            self.col.learn(("pfc",), self.ctx(g, act), op)\n            g["state"] = kind\n\n    def attended(self):\n        b = self.col.keys.get(("pfc",))\n        return None if b is None else b.attended()\n', 'spotlight.py': '"""Spotlight memory (ARCHITECTURE.md section 4): a lattice of cells at addresses, kept apart from the models that use it.\n\nTaken from Percepta\'s Spotlight: memory is a lattice, it grows without changing any model, and a cell is read and\nwritten at its address, or through a neighbourhood when the address is uncertain. Changed for this agent (COORDINATES\npart 5): addresses are computed and path-integrated by a column\'s reference frame, not learned by projecting a key,\nso exact reads are the normal case.\n\nA cell is a small associative memory updated by the delta rule with one-hot keys (DeltaNet\'s update in its simplest\nform): for each key, w <- w + beta * (onehot(value) - w). With beta = 1 it holds the last value written; with beta < 1\nit holds a distribution over values.\n"""\n\n\nclass Spotlight:\n    def __init__(self, beta=1.0):\n        self.beta = beta\n        self.cells = {}                                 # address -> {key: {value: weight}}\n\n    def write(self, addr, key, value, beta=None):\n        b = self.beta if beta is None else beta\n        w = self.cells.setdefault(addr, {}).setdefault(key, {})\n        for v in list(w):\n            w[v] *= 1.0 - b\n            if w[v] < 1e-6:\n                del w[v]\n        w[value] = w.get(value, 0.0) + b\n\n    def read(self, addr, key):\n        """{value: weight} at an exact address, or None if nothing was written there."""\n        cell = self.cells.get(addr)\n        return None if cell is None else cell.get(key)\n\n    def value(self, addr, key, default=None):\n        w = self.read(addr, key)\n        return max(w, key=w.get) if w else default\n\n    def read_soft(self, belief, key):\n        """The belief-weighted mixture of the cells at the believed addresses ({address: probability})."""\n        out = {}\n        for addr, p in belief.items():\n            for v, w in (self.read(addr, key) or {}).items():\n                out[v] = out.get(v, 0.0) + p * w\n        return out\n\n    def forget(self, addrs=None):\n        if addrs is None:\n            self.cells.clear()\n        else:\n            for a in addrs:\n                self.cells.pop(a, None)\n\n    def __len__(self):\n        return len(self.cells)\n', 'symreg.py': '"""The function finder, version 2 (legacy/docs/CURRICULUM.md, the experiments that built it; a copy for the agent): terms with continuous parameters, safety over the design\ndomain, selection by BIC with a library prior, and a frontier of near-equal formulas as an ensemble.\n\nA formula is t = (c0 + sum_k B_k(x; theta_k) . c_k) / (1 + D(x)), with t = y or log y. A block B_k is a product of\nfactors and gives one column, or one column per input of a group (shared parameters, one coefficient each).\nFactor kinds and their parameters:\n    pow     x_i^p                        p\n    shift   (x_i + b)^p                  beta, p      (b = -min x_i + range * (0.1 + softplus(beta)): the base stays\n                                                       positive on the box, and a singularity sits at least 10% of the\n                                                       range below it, so modest extrapolation can\'t reach it)\n    margin  (s_j + b)^p, b > 0           beta, p      (s_j a constraint margin, >= 0 on the domain)\n    log     log x_i                      -\n    exp     exp(k z_i), z_i in [-1, 1]   k\n    den     -t * prod x_i^m (integers, degree 1-2): a denominator column (no parameters)\nSearch (polynomial): grow one block at a time on the residual. Each step screens every candidate on a parameter grid\nby the exact drop in squared error after a linear refit, and refines the best REFINE candidates\' own parameters by\nbounded least squares (variable projection: the linear coefficients are solved exactly inside; at most LM_EVALS\nresidual evaluations, finite-difference Jacobians included). The BEAM best formulas of each size then get a joint\nrefit of every free parameter, under the same budget.\nSafety: every block is evaluated on the task\'s domain sample (feasible designs, with boundary points) and rejected\nif non-finite or if its largest magnitude exceeds 1e4 x its median; a denominator must stay above 0.05 there.\nSelection: BIC on the y scale, n log(RSS/n) + parameters * log n (+ 2 sum log y for a log-y formula)\n- 2 sum log P(block template); or, given the noise level sigma (experiment B2), BIC with sigma known:\nsum (y_hat - y)^2 / sigma^2 + parameters * log n - 2 sum log P, residuals on the y scale for both transforms.\nFrontier: every formula within 6 BIC of the best, at most 8, weighted exp(-BIC/2).\n\nVersion 3 (experiment B3, docs/VOCABULARY.md), each behind a flag:\n    outer        composition by inversion, depth 2: y = f(T(x)); the inner sum T is searched on t = f^-1(y). The family:\n                 y; log(y - b); (y - b)^q, q in {-1, -1/2, 1/2, 2}; b in {0, min y - 0.05 range y}. Every member is\n                 scored by how well a short sum fits its target: a cheap 3-term greedy screen (grid starts, no\n                 refinement), measured on the y scale; the top 3, plus y and log y, are searched. (Amended before any\n                 run: first-order additivity, as first specified, can\'t discriminate in 1-D and penalizes products.)\n    affine       atoms over a fitted combination of inputs: (a.u + b)^p and log(a.u + b), u the inputs scaled to the\n                 box, a a direction, b keeping the base positive with any singularity 10% of its range outside.\n    relational   one term x_i^p x_j^q summed over a set of input pairs (shared p, q; a coefficient per pair), over all\n                 pairs and over the pairs the guide detects as interacting.\n"""\nimport itertools\nimport math\n\nimport numpy as np\nfrom scipy.optimize import least_squares\n\nGRID = (-3.0, -2.5, -2.0, -1.5, -1.0, -0.5, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0)\nEXP_GRID = (-2.0, -1.0, 1.0, 2.0)\nBETA_GRID = (-2.0, 0.0)\nBEAM, K_MAX, REFINE, LM_EVALS, FRONTIER, FRONTIER_BIC = 4, 8, 8, 50, 8, 6.0\nK_CONTINUE = 4                     # blocks the search may add after a start formula (the guide\'s assembled 1-D fits)\nPRIMITIVES = ("pow", "shift", "log", "exp", "pow*pow", "margin", "den", "affine", "alog", "pairgroup")\nAFFINE_P = (-2.0, -1.0, -0.5, 0.5, 1.0, 2.0)\n\n\n# ---- outer functions (B3) --------------------------------------------------------------------------------------------\ndef as_transform(tr):\n    return ("identity",) if tr == "identity" else ("log", 0.0) if tr == "log" else tuple(tr)\n\n\ndef t_forward(tr, y):\n    tr = as_transform(tr)\n    with np.errstate(all="ignore"):\n        if tr[0] == "identity":\n            return y\n        if tr[0] == "log":\n            return np.log(y - tr[1])\n        return (y - tr[2]) ** tr[1]\n\n\ndef t_inverse(tr, t):\n    tr = as_transform(tr)\n    with np.errstate(all="ignore"):\n        if tr[0] == "identity":\n            return t\n        if tr[0] == "log":\n            return np.exp(np.clip(t, -50, 50)) + tr[1]\n        return np.power(t, 1.0 / tr[1]) + tr[2]\n\n\ndef t_valid(tr, y):\n    tr = as_transform(tr)\n    if tr[0] == "identity":\n        return True\n    b = tr[1] if tr[0] == "log" else tr[2]\n    return bool(np.all(y - b > 0))\n\n\ndef t_label(tr):\n    tr = as_transform(tr)\n    if tr[0] == "identity":\n        return "y"\n    b = tr[1] if tr[0] == "log" else tr[2]\n    yb = "y" if b == 0 else f"(y{-b:+.4g})"\n    return f"log {yb}" if tr[0] == "log" else f"{yb}^{tr[1]:g}"\n\n\ndef outer_family(y):\n    bs = [0.0, float(np.min(y) - 0.05 * (np.max(y) - np.min(y)))]\n    fam = [("identity",)] + [("log", b) for b in bs] + [("pow", q, b) for q in (-1.0, -0.5, 0.5, 2.0) for b in bs]\n    return [tr for tr in fam if t_valid(tr, y)]\nN_PARAMS = {"pow": 1, "shift": 2, "margin": 2, "log": 0, "exp": 1}\nP_BOUND, P_MIN = 6.0, 0.25\n\n\ndef softplus(v):\n    return np.log1p(np.exp(-abs(v))) + max(v, 0.0)\n\n\nclass Context:\n    """The data a formula is evaluated on: inputs, margins, and the per-input offsets that keep shifted bases positive."""\n    def __init__(self, X, margins, domain):\n        self.lo, self.hi = domain["X"].min(0), domain["X"].max(0)\n        self.X, self.M = X, margins\n        self.dom_X, self.dom_M = domain["X"], domain["M"]\n        self.mlo = domain["M"].min(0) if domain["M"].size else np.zeros(0)\n        self.mrange = (domain["M"].max(0) - self.mlo) if domain["M"].size else np.zeros(0)\n\n\ndef shift_offset(ctx, var, beta):\n    return -ctx.lo[var] + (ctx.hi[var] - ctx.lo[var]) * (0.1 + softplus(beta))\n\n\ndef margin_offset(ctx, var, beta):\n    # b > 0 from the constraint boundary s = 0, so every feasible design (s >= 0) has a positive base (fixed\n    # 2026-10-26: it was measured from the training domain\'s smallest margin, which extrapolation can undercut)\n    return ctx.mrange[var] * (1e-3 + softplus(beta))\n\n\ndef factor_value(kind, var, params, X, M, ctx):\n    if kind == "pow":\n        return X[:, var] ** params[0]\n    if kind == "shift":\n        rng = ctx.hi[var] - ctx.lo[var]\n        return (X[:, var] + shift_offset(ctx, var, params[0])) ** params[1]\n    if kind == "margin":\n        return (M[:, var] + margin_offset(ctx, var, params[0])) ** params[1]\n    if kind == "log":\n        return np.log(X[:, var])\n    if kind == "exp":\n        z = 2 * (X[:, var] - ctx.lo[var]) / (ctx.hi[var] - ctx.lo[var]) - 1\n        return np.exp(params[0] * z)\n    raise ValueError(kind)\n\n\nclass Block:\n    """A product of factors. factors: [(kind, slot)], slots index into each binding; bindings: [tuple of input or\n    margin indices], one column per binding (one binding = an ordinary term, several = a group). theta: all factor\n    parameters; free: which are fitted. template: the library entry it came from (for the prior)."""\n    def __init__(self, factors, bindings, theta, free, template):\n        self.factors, self.bindings = list(factors), [tuple(b) for b in bindings]\n        self.theta, self.free, self.template = np.asarray(theta, float), np.asarray(free, bool), template\n\n    def copy(self):\n        return Block(self.factors, self.bindings, self.theta.copy(), self.free.copy(), self.template)\n\n    def columns(self, X, M, ctx, theta=None):\n        theta = self.theta if theta is None else theta\n        cols = []\n        for bind in self.bindings:\n            v, at = np.ones(len(X)), 0\n            for kind, slot in self.factors:\n                k = N_PARAMS[kind]\n                v = v * factor_value(kind, bind[slot], theta[at:at + k], X, M, ctx)\n                at += k\n            cols.append(v)\n        return np.stack(cols, 1)\n\n    def n_free(self):\n        return int(self.free.sum())\n\n    def beta_mask(self):\n        roles = []\n        for kind, _ in self.factors:\n            roles += (["beta", "p"] if kind in ("shift", "margin") else ["p"] * N_PARAMS[kind])\n        return np.array([r == "beta" for r in roles], bool)\n\n    def exponent_mask(self):\n        roles = []\n        for kind, _ in self.factors:\n            roles += {"pow": [True], "shift": [False, True], "margin": [False, True], "log": [], "exp": [True]}[kind]\n        return np.array(roles, bool)\n\n    def key(self):\n        return (tuple(self.factors), tuple(self.bindings), tuple(np.round(self.theta[~self.free], 3)))\n\n    def safe(self, ctx):\n        with np.errstate(all="ignore"):\n            C = self.columns(ctx.dom_X, ctx.dom_M, ctx)\n        if not np.all(np.isfinite(C)):\n            return False\n        A = np.abs(C)\n        return bool(np.all(A.max(0) <= 1e4 * np.maximum(np.median(A, 0), 1e-300)))\n\n    def label(self, names, mnames, ctx=None):\n        parts, at = [], 0\n        for kind, slot in self.factors:\n            k = N_PARAMS[kind]\n            p = self.theta[at:at + k]\n            at += k\n            v = "{" + ",".join(names[b[slot]] if kind != "margin" else mnames[b[slot]] for b in self.bindings) + "}" \\\n                if len(self.bindings) > 1 else (mnames[self.bindings[0][slot]] if kind == "margin" else names[self.bindings[0][slot]])\n            off = lambda f: f(ctx, self.bindings[0][slot], p[0]) if ctx is not None else float("nan")\n            parts.append({"pow": lambda: f"{v}^{p[0]:.3g}", "shift": lambda: f"({v}{off(shift_offset):+.3g})^{p[1]:.3g}",\n                          "margin": lambda: f"({v}{off(margin_offset):+.3g})^{p[1]:.3g}", "log": lambda: f"log({v})",\n                          "exp": lambda: f"exp({p[0]:.3g} z_{v})"}[kind]())\n        return ("sum over " if len(self.bindings) > 1 else "") + "*".join(parts)\n\n\nclass AffineBlock(Block):\n    """(a.u + b)^p or log(a.u + b) over all inputs (B3): u = inputs scaled to [0, 1] over the box, a a direction\n    (normalized inside), b = -min(a.u) + (0.1 + softplus(beta)) range(a.u) over the domain. Params: a (d), beta, p."""\n    def __init__(self, kind, d, theta, free=None, template=None):\n        k = d + (2 if kind == "affine" else 1)\n        super().__init__([(kind, 0)], [(0,)], theta, np.ones(k, bool) if free is None else free, template or kind)\n        self.kind, self.d = kind, d\n\n    def copy(self):\n        return AffineBlock(self.kind, self.d, self.theta.copy(), self.free.copy(), self.template)\n\n    def base(self, X, ctx, theta):\n        a = np.asarray(theta[:self.d], float)\n        nrm = np.linalg.norm(a)\n        if nrm < 1e-9:\n            return None\n        a = a / nrm\n        span = np.where(ctx.hi > ctx.lo, ctx.hi - ctx.lo, 1.0)\n        s, sd = ((X - ctx.lo) / span) @ a, ((ctx.dom_X - ctx.lo) / span) @ a\n        rng = max(float(sd.max() - sd.min()), 1e-9)\n        return s - sd.min() + (0.1 + softplus(theta[self.d])) * rng\n\n    def columns(self, X, M, ctx, theta=None):\n        theta = self.theta if theta is None else theta\n        bs = self.base(X, ctx, theta)\n        if bs is None:\n            return np.full((len(X), 1), np.nan)\n        with np.errstate(all="ignore"):\n            v = bs ** theta[self.d + 1] if self.kind == "affine" else np.log(bs)\n        return v[:, None]\n\n    def beta_mask(self):\n        m = np.zeros(len(self.theta), bool)\n        m[self.d] = True\n        return m\n\n    def exponent_mask(self):\n        m = np.zeros(len(self.theta), bool)\n        if self.kind == "affine":\n            m[self.d + 1] = True\n        return m\n\n    def label(self, names, mnames, ctx=None):\n        a = self.theta[:self.d] / max(np.linalg.norm(self.theta[:self.d]), 1e-9)\n        comb = "".join(f"{c:+.2g}*u_{n}" for c, n in zip(a, names) if abs(c) > 0.05)\n        return f"({comb}+b)^{self.theta[self.d + 1]:.3g}" if self.kind == "affine" else f"log({comb}+b)"\n\n\nclass State:\n    """A formula: blocks, denominator monomials, fitted coefficients."""\n    def __init__(self, blocks, dens):\n        self.blocks, self.dens = blocks, list(dens)\n\n    def copy(self):\n        return State([b.copy() for b in self.blocks], self.dens)\n\n    def key(self):\n        return (tuple(sorted(b.key() for b in self.blocks)), tuple(sorted(self.dens)))\n\n\ndef den_cols(dens, X, t):\n    return np.stack([-t * np.prod(X ** np.array(m, float), 1) for m in dens], 1) if dens else np.zeros((len(X), 0))\n\n\nclass Finder:\n    def __init__(self, names, mnames=(), library=None, prior=None, k_max=K_MAX, outer=False, affine=False,\n                 relational=False):\n        """library: list of template specs (see library.py); None = primitives only. prior: {primitive or template name:\n        weight}, default 1 each; P(name) = weight / total."""\n        self.names, self.mnames, self.k_max = list(names), list(mnames), k_max\n        self.library = library or []\n        self.prior = dict(prior or {})\n        self.outer, self.affine, self.relational = outer, affine, relational\n\n    # ---- candidates ---------------------------------------------------------------------------------------------------\n    def primitive_candidates(self, d, n_margins, hints):\n        """(Block with initial theta, list of initial thetas for its free parameters)."""\n        out = []\n        extra = hints.get("exponents", {})\n        for i in range(d):\n            grid = sorted(set(GRID) | {round(float(e), 2) for e in extra.get(i, [])})\n            out.append((Block([("pow", 0)], [(i,)], [1.0], [True], "pow"), [[p] for p in grid]))\n            out.append((Block([("shift", 0)], [(i,)], [0.0, 1.0], [True, True], "shift"),\n                        [[bb, p] for bb in BETA_GRID for p in GRID if p < 0]))\n            out.append((Block([("log", 0)], [(i,)], [], [], "log"), [[]]))\n            out.append((Block([("exp", 0)], [(i,)], [1.0], [True], "exp"), [[k] for k in EXP_GRID]))\n        for i, j in itertools.combinations(range(d), 2):\n            out.append((Block([("pow", 0), ("pow", 1)], [(i, j)], [1.0, 1.0], [True, True], "pow*pow"),\n                        [[p, q] for p in GRID for q in GRID if abs(p) + abs(q) <= 6]))\n        for j in range(n_margins):\n            out.append((Block([("margin", 0)], [(j,)], [0.0, -1.0], [True, True], "margin"),\n                        [[bb, p] for bb in BETA_GRID for p in GRID if p < 0]))\n        if self.affine and d >= 2:\n            dirs = [np.eye(d)[i] + s * np.eye(d)[j] for i, j in itertools.combinations(range(d), 2) for s in (1, -1)]\n            dirs.append(np.ones(d))\n            for kind in ("affine", "alog"):\n                inits = [list(a) + [bb] + ([p] if kind == "affine" else []) for a in dirs for bb in BETA_GRID\n                         for p in (AFFINE_P if kind == "affine" else [None])]\n                out.append((AffineBlock(kind, d, inits[0]), inits))\n        if self.relational and d >= 2:\n            pair_sets = [list(itertools.combinations(range(d), 2))]\n            det = [tuple(p) for p in hints.get("pairs", [])]\n            if len(det) >= 2 and det != pair_sets[0]:\n                pair_sets.append(det)\n            for ps in pair_sets:\n                if len(ps) >= 2:\n                    out.append((Block([("pow", 0), ("pow", 1)], ps, [1.0, 1.0], [True, True], "pairgroup"),\n                                [[p, q] for p in GRID for q in GRID if abs(p) + abs(q) <= 6]))\n        return out\n\n    def library_candidates(self, d, hints):\n        """Every library template, bound to every ordered choice of distinct inputs for its slots, or, for a group\n        template, to each group the guide found and to all inputs. Free parameters start on their grids."""\n        out = []\n        for tpl in self.library:\n            s, free = tpl["slots"], np.asarray(tpl["free"], bool)\n            probe = Block(tpl["factors"], [tuple(range(s))], tpl["theta"], free, tpl["name"])\n            beta = probe.beta_mask()\n            rate = np.array([kind == "exp" for kind, _ in probe.factors for _ in range(N_PARAMS[kind])], bool)\n            grids = [BETA_GRID if beta[k] else EXP_GRID if rate[k] else GRID for k in np.nonzero(free)[0]]\n            inits = [list(p) for p in itertools.product(*grids)][:64] if free.any() else [[]]\n            if tpl.get("group") and s == 2:                      # a relational template: sets of input pairs\n                sets = [[tuple(p) for p in hints.get("pairs", [])], list(itertools.combinations(range(d), 2))]\n                bindings_list = [ps for ps in sets if len(ps) >= 2]\n            elif tpl.get("group"):\n                groups = [list(g) for g in hints.get("groups", []) if len(g) >= 2] + ([list(range(d))] if d >= 2 else [])\n                bindings_list = [[(v,) for v in g] for g in groups]\n            else:\n                bindings_list = [[b] for b in itertools.permutations(range(d), s)] if s <= d else []\n            for bindings in bindings_list:\n                out.append((Block(tpl["factors"], bindings, tpl["theta"], free, tpl["name"]), inits))\n        return out\n\n    def den_candidates(self, d):\n        return [m for m in itertools.product(range(3), repeat=d) if 1 <= sum(m) <= 2]\n\n    # ---- fitting ------------------------------------------------------------------------------------------------------\n    def design(self, state, X, M, t, ctx, thetas=None):\n        cols = [np.ones((len(X), 1))]\n        for k, b in enumerate(state.blocks):\n            cols.append(b.columns(X, M, ctx, None if thetas is None else thetas[k]))\n        cols.append(den_cols(state.dens, X, t))\n        return np.column_stack(cols)\n\n    def fit_linear(self, state, X, M, t, ctx, thetas=None):\n        with np.errstate(all="ignore"):\n            A = self.design(state, X, M, t, ctx, thetas)\n        if not np.all(np.isfinite(A)):\n            return None, np.inf\n        coef, *_ = np.linalg.lstsq(A, t, rcond=None)\n        r = t - A @ coef\n        return coef, float(r @ r)\n\n    def refit_joint(self, state, X, M, t, ctx, only_last=False):\n        """Refine the free parameters (all blocks, or only the last) by variable projection, bounded, at most\n        LM_EVALS residual evaluations including the finite-difference Jacobian\'s."""\n        free = [b for b in (state.blocks[-1:] if only_last else state.blocks) if b.n_free()]\n        if not free:\n            return self.fit_linear(state, X, M, t, ctx)\n        x0 = np.concatenate([b.theta[b.free] for b in free])\n        beta = np.concatenate([b.beta_mask()[b.free] for b in free])\n        expo = np.concatenate([b.exponent_mask()[b.free] for b in free])\n        lo = np.where(beta, -10.0, -P_BOUND)\n        hi = np.where(beta, 5.0, P_BOUND)\n        # an exponent or rate keeps its sign and stays >= P_MIN from zero (c x^eps fakes log x, c exp(eps z) fakes z)\n        lo = np.where(expo & (x0 > 0), P_MIN, lo)\n        hi = np.where(expo & (x0 <= 0), -P_MIN, hi)\n        x0 = np.clip(x0, lo + 1e-9, hi - 1e-9)\n\n        def unpack(z):\n            out, at = [], 0\n            for b in state.blocks:\n                th = b.theta.copy()\n                if b.n_free() and any(b is f for f in free):\n                    th[b.free] = z[at:at + b.n_free()]\n                    at += b.n_free()\n                out.append(th)\n            return out\n\n        def resid(z):\n            th = unpack(z)\n            with np.errstate(all="ignore"):\n                A = self.design(state, X, M, t, ctx, th)\n            if not np.all(np.isfinite(A)):\n                return np.full(len(t), 1e6)\n            coef, *_ = np.linalg.lstsq(A, t, rcond=None)\n            return t - A @ coef\n\n        try:\n            sol = least_squares(resid, x0, bounds=(lo, hi), max_nfev=max(2, LM_EVALS // (1 + len(x0))))\n            z = sol.x\n        except (ValueError, np.linalg.LinAlgError):\n            z = x0\n        for b, th in zip(state.blocks, unpack(z)):\n            b.theta = th\n        return self.fit_linear(state, X, M, t, ctx)\n\n    @staticmethod\n    def _is_exponent(block):\n        return block.exponent_mask()\n\n    @staticmethod\n    def _is_beta(block):\n        return block.beta_mask()\n\n    def screen(self, state, cands, X, M, t, ctx):\n        """Score every (candidate, initial theta) by the drop in squared error after a linear refit."""\n        with np.errstate(all="ignore"):\n            A0 = self.design(state, X, M, t, ctx)\n        Q, _ = np.linalg.qr(A0)\n        r = t - Q @ (Q.T @ t)\n        scored = []\n        for ci, (block, inits) in enumerate(cands):\n            for init in inits:\n                th = block.theta.copy()\n                if block.n_free():\n                    th[block.free] = init\n                with np.errstate(all="ignore"):\n                    C = block.columns(X, M, ctx, th)\n                if not np.all(np.isfinite(C)):\n                    continue\n                Cp = C - Q @ (Q.T @ C)\n                norm = np.linalg.norm(Cp, axis=0)\n                if np.any(norm < 1e-10 * max(np.linalg.norm(C), 1e-300)):\n                    continue\n                c, *_ = np.linalg.lstsq(Cp, r, rcond=None)\n                gain = float(np.sum((Cp @ c) ** 2))\n                scored.append((gain, ci, init))\n        scored.sort(key=lambda s: -s[0])\n        return scored\n\n    def bic(self, state, rss, n, transform, y):\n        # a numerical floor, the same precision on the y scale for both transforms (exact data can\'t overfit)\n        floor = 1e-12 * float(np.sum((y - y.mean()) ** 2))\n        is_log = as_transform(transform) == ("log", 0.0)\n        rss = max(rss, floor / float(np.exp(2 * np.mean(np.log(y)))) if is_log else floor)\n        p = 1 + sum(len(b.bindings) + b.n_free() for b in state.blocks) + len(state.dens)\n        jac = 2 * float(np.sum(np.log(y))) if is_log else 0.0\n        prior = sum(math.log(self.prior_prob(b.template)) for b in state.blocks) + len(state.dens) * math.log(self.prior_prob("den"))\n        return n * math.log(max(rss, 1e-300) / n) + p * math.log(n) + jac - 2 * prior\n\n    def prior_prob(self, name):\n        names = list(PRIMITIVES) + [t["name"] for t in self.library]\n        total = sum(self.prior.get(k, 1.0) for k in names)\n        return self.prior.get(name, 1.0) / total\n\n    def pole_free(self, state, coef, ctx):\n        if not state.dens:\n            return True\n        m = len(state.dens)\n        D = np.stack([np.prod(ctx.dom_X ** np.array(mm, float), 1) for mm in state.dens], 1) @ coef[-m:]\n        return bool(np.min(1 + D) > 0.05)\n\n    # ---- the search ---------------------------------------------------------------------------------------------------\n    def fit(self, X, y, margins=None, domain=None, hints=None, start=None, start_transform=None, transforms=None,\n            noise=None):\n        """The plain search from an empty formula, for each transform (default y and, if positive, log y); and, if a\n        start formula is given, a further search from it in start_transform, adding at most K_CONTINUE blocks. Every\n        formula met goes into one pool, from which BIC picks the frontier."""\n        X, y = np.asarray(X, float), np.asarray(y, float)\n        n, d = X.shape\n        M = np.zeros((n, 0)) if margins is None else np.asarray(margins, float)\n        self.noise, self._train = noise, (X, M)\n        domain = domain or {"X": X, "M": M}\n        ctx = Context(X, M, domain)\n        hints = hints or {}\n        safe_cache = {}\n        cands = self.primitive_candidates(d, M.shape[1], hints) + self.library_candidates(d, hints)\n        dens = self.den_candidates(d)\n        if transforms is None and self.outer:\n            scores = {}\n            for tr in outer_family(y):\n                t = t_forward(tr, y)\n                if np.all(np.isfinite(t)) and np.std(t) > 0:\n                    scores[tr] = self.quick_score(tr, X, M, y, t, ctx, cands)\n            self.outer_scores = scores\n            top = sorted(scores, key=scores.get)[:3]\n            transforms = [("identity",)] + ([("log", 0.0)] if ("log", 0.0) in scores else []) + top\n        transforms = list(dict.fromkeys(as_transform(tr) for tr in (transforms or\n                                        (["identity"] + (["log"] if np.all(y > 0) else [])))))\n        runs = [(tr, None) for tr in transforms]\n        if start is not None:\n            runs.append((as_transform(start_transform), start))\n        self.records = []\n        for transform, init in runs:\n            t = t_forward(transform, y)\n            beam = [init.copy() if init is not None else State([], [])]\n            if init is not None:\n                coef, rss = self.refit_joint(beam[0], X, M, t, ctx)\n                steps = max(0, min(K_CONTINUE, (n - 2) // 2 - len(init.blocks)))\n            else:\n                coef, rss = self.fit_linear(beam[0], X, M, t, ctx)\n                steps = max(1, min(self.k_max, (n - 2) // 2))\n            self._record(beam[0], coef, rss, transform, y, n, ctx)\n            for _ in range(steps):\n                new = {}\n                for st in beam:\n                    scored = self.screen(st, cands, X, M, t, ctx)\n                    tried = set()                                      # refine the REFINE best distinct candidates\n                    for gain, ci, init in scored:\n                        if len(tried) >= REFINE:\n                            break\n                        if ci in tried:\n                            continue\n                        block = cands[ci][0].copy()\n                        if block.n_free():\n                            block.theta[block.free] = init\n                        nxt = State([b.copy() for b in st.blocks] + [block], st.dens)\n                        coef, rss = self.refit_joint(nxt, X, M, t, ctx, only_last=True)\n                        if coef is None:\n                            continue\n                        tried.add(ci)\n                        if not all(self._safe(b, ctx, safe_cache) for b in nxt.blocks) or not self.pole_free(nxt, coef, ctx):\n                            continue\n                        k = nxt.key()\n                        if k not in new or rss < new[k][2]:\n                            new[k] = (nxt, coef, rss)\n                    for m in dens[:0] if len(st.dens) >= 2 else dens:          # denominators, at most 2\n                        nxt = State([b.copy() for b in st.blocks], st.dens + [m])\n                        coef, rss = self.fit_linear(nxt, X, M, t, ctx)\n                        if coef is None or not self.pole_free(nxt, coef, ctx):\n                            continue\n                        k = nxt.key()\n                        if k not in new or rss < new[k][2]:\n                            new[k] = (nxt, coef, rss)\n                if not new:\n                    break\n                ranked = sorted(new.values(), key=lambda v: v[2])[:BEAM]\n                beam = []\n                for st_, coef_, rss_ in ranked:\n                    trial = st_.copy()\n                    c2, r2 = self.refit_joint(trial, X, M, t, ctx)\n                    if c2 is not None and r2 < rss_ and all(self._safe(b, ctx, safe_cache) for b in trial.blocks)                             and self.pole_free(trial, c2, ctx):\n                        st_, coef_, rss_ = trial, c2, r2\n                    beam.append(st_)\n                    self._record(st_, coef_, rss_, transform, y, n, ctx)\n        self._finish(n)\n        return self\n\n    def quick_score(self, tr, X, M, y, t, ctx, cands, terms=3):\n        """How well a short sum fits the target t = f^-1(y): greedy screening for `terms` blocks at their grid starts\n        (no refinement), the y-scale residual sum of squares of the result (lower is better)."""\n        st = State([], [])\n        for _ in range(terms):\n            scored = self.screen(st, cands, X, M, t, ctx)\n            if not scored:\n                break\n            _, ci, init = scored[0]\n            b = cands[ci][0].copy()\n            if b.n_free():\n                b.theta[b.free] = init\n            st = State(st.blocks + [b], [])\n        coef, _ = self.fit_linear(st, X, M, t, ctx)\n        if coef is None:\n            return np.inf\n        pred = t_inverse(tr, self.design(st, X, M, t, ctx) @ coef)\n        return float(np.sum((pred - y) ** 2)) if np.all(np.isfinite(pred)) else np.inf\n\n    def _safe(self, block, ctx, cache):\n        k = (block.key(), tuple(np.round(block.theta, 6)))\n        if k not in cache:\n            cache[k] = block.safe(ctx)\n        return cache[k]\n\n    def _record(self, state, coef, rss, transform, y, n, ctx):\n        if coef is None or not self.pole_free(state, coef, ctx):\n            return\n        rec = dict(state=state.copy(), coef=coef, rss=rss, transform=transform, ctx=ctx)\n        if self.noise is None:\n            rec["bic"] = self.bic(state, rss, n, transform, y)\n        else:\n            pred = self.predict_one(rec, *self._train)\n            rss_y = float(np.sum((pred - y) ** 2)) if np.all(np.isfinite(pred)) else np.inf\n            p = 1 + sum(len(b.bindings) + b.n_free() for b in state.blocks) + len(state.dens)\n            prior = sum(math.log(self.prior_prob(b.template)) for b in state.blocks) + \\\n                len(state.dens) * math.log(self.prior_prob("den"))\n            rec["bic"] = rss_y / self.noise ** 2 + p * math.log(n) - 2 * prior\n        if np.isfinite(rec["bic"]):\n            self.records.append(rec)\n\n    def _finish(self, n):\n        recs = sorted(self.records, key=lambda r: r["bic"])\n        seen, front = set(), []\n        for r in recs:\n            k = (r["state"].key(), r["transform"])\n            if k in seen:\n                continue\n            seen.add(k)\n            if r["bic"] <= recs[0]["bic"] + FRONTIER_BIC and len(front) < FRONTIER:\n                front.append(r)\n        b0 = front[0]["bic"]\n        w = np.array([math.exp(-(r["bic"] - b0) / 2) for r in front])\n        self.frontier, self.weights = front, w / w.sum()\n        self.best = front[0]\n\n    # ---- prediction ---------------------------------------------------------------------------------------------------\n    def predict_one(self, rec, X, margins=None):\n        X = np.asarray(X, float)\n        M = np.zeros((len(X), 0)) if margins is None else np.asarray(margins, float)\n        st, coef = rec["state"], rec["coef"]\n        m = len(st.dens)\n        with np.errstate(all="ignore"):\n            A = self.design(State(st.blocks, []), X, M, np.zeros(len(X)), rec["ctx"])\n            P = A @ coef[:A.shape[1]]\n            D = (np.stack([np.prod(X ** np.array(mm, float), 1) for mm in st.dens], 1) @ coef[-m:]) if m else 0.0\n            t = P / (1 + D)\n            return t_inverse(rec["transform"], t)\n\n    def predict(self, X, margins=None):\n        preds = np.stack([self.predict_one(r, X, margins) for r in self.frontier])\n        return self.weights @ preds\n\n    def spread(self, X, margins=None, scale=1.0):\n        preds = np.stack([self.predict_one(r, X, margins) for r in self.frontier])\n        mean = self.weights @ preds\n        return float(np.mean(np.sqrt(self.weights @ (preds - mean) ** 2)) / scale)\n\n    def formula(self, rec=None):\n        rec = rec or self.best\n        st, coef = rec["state"], rec["coef"]\n        terms, at = [f"{coef[0]:.4g}"], 1\n        for b in st.blocks:\n            k = len(b.bindings)\n            c = coef[at:at + k]\n            at += k\n            terms.append((f"{c[0]:+.4g}*" if k == 1 else "+[" + ",".join(f"{x:.3g}" for x in c) + "]*") + b.label(self.names, self.mnames, rec["ctx"]))\n        body = " ".join(terms)\n        if st.dens:\n            dc = coef[at:]\n            den = " ".join(f"{c:+.3g}*" + "*".join(f"{self.names[i]}^{e}" for i, e in enumerate(m) if e) for c, m in zip(dc, st.dens))\n            body = f"({body}) / (1 {den})"\n        return f"{t_label(rec[\'transform\'])} = " + body\n\n    def n_terms(self, rec=None):\n        rec = rec or self.best\n        return 1 + sum(len(b.bindings) for b in rec["state"].blocks) + len(rec["state"].dens)\n', 'priors.json': '{\n "draws": [\n  "joint",\n  "on-rare"\n ],\n "kinds": [\n  "own state",\n  "numeric",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "unconditional",\n  "other column",\n  "unconditional",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "elsewhere",\n  "other column",\n  "numeric",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "elsewhere",\n  "other column",\n  "numeric",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "other column",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "unconditional",\n  "unconditional",\n  "unconditional",\n  "unconditional",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "ahead",\n  "own state",\n  "other column",\n  "own state",\n  "ahead",\n  "own state",\n  "other column",\n  "ahead",\n  "other column",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "elsewhere",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "unconditional",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "numeric",\n  "other column",\n  "other column",\n  "unconditional",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "unconditional",\n  "own state",\n  "own state",\n  "unconditional",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "unconditional",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "ahead",\n  "own state",\n  "conjunction",\n  "unconditional",\n  "unconditional",\n  "conjunction",\n  "unconditional",\n  "elsewhere",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "elsewhere",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "ahead",\n  "elsewhere",\n  "ahead",\n  "other column",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "ahead",\n  "elsewhere",\n  "ahead",\n  "elsewhere",\n  "own state",\n  "ahead",\n  "numeric",\n  "ahead",\n  "other column",\n  "own state",\n  "other column",\n  "elsewhere",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "ahead",\n  "unconditional",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "elsewhere",\n  "ahead",\n  "ahead",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "ahead",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "ahead",\n  "other column",\n  "other column",\n  "own state",\n  "other column",\n  "ahead",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "ahead",\n  "own state",\n  "own state",\n  "own state",\n  "ahead",\n  "ahead",\n  "ahead",\n  "own state",\n  "ahead",\n  "ahead",\n  "ahead",\n  "own state",\n  "ahead",\n  "ahead",\n  "ahead",\n  "ahead",\n  "ahead",\n  "other column",\n  "own state",\n  "numeric",\n  "ahead",\n  "elsewhere",\n  "elsewhere",\n  "elsewhere",\n  "elsewhere",\n  "other column",\n  "conjunction",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "other column",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "other column",\n  "unconditional",\n  "other column",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "own state",\n  "own state",\n  "own state",\n  "other column",\n  "other column",\n  "other column"\n ],\n "library": [\n  {\n   "name": "T0",\n   "factors": [\n    [\n     "pow",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    0.5139176327585924\n   ],\n   "free": [\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T1",\n   "factors": [\n    [\n     "exp",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    5.979231956517838\n   ],\n   "free": [\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T2",\n   "factors": [\n    [\n     "pow",\n     0\n    ],\n    [\n     "pow",\n     1\n    ]\n   ],\n   "slots": 2,\n   "theta": [\n    -0.5102197005302959,\n    4.011273465305397\n   ],\n   "free": [\n    true,\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T3",\n   "factors": [\n    [\n     "shift",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    -1.4525146445027397,\n    -5.955230041855765\n   ],\n   "free": [\n    true,\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T4",\n   "factors": [\n    [\n     "pow",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    5.9999999999991775\n   ],\n   "free": [\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T5",\n   "factors": [\n    [\n     "exp",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    -5.999999376935977\n   ],\n   "free": [\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T6",\n   "factors": [\n    [\n     "pow",\n     0\n    ],\n    [\n     "pow",\n     1\n    ]\n   ],\n   "slots": 2,\n   "theta": [\n    0.5125641321243473,\n    2.0\n   ],\n   "free": [\n    false,\n    true\n   ],\n   "group": false\n  },\n  {\n   "name": "T7",\n   "factors": [\n    [\n     "shift",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    0.0,\n    -1.0\n   ],\n   "free": [\n    false,\n    true\n   ],\n   "group": false\n  },\n  {\n   "name": "T8",\n   "factors": [\n    [\n     "exp",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    -1.0\n   ],\n   "free": [\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T9",\n   "factors": [\n    [\n     "shift",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    -2.0,\n    -2.0\n   ],\n   "free": [\n    false,\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T10",\n   "factors": [\n    [\n     "exp",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    1.0\n   ],\n   "free": [\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T11",\n   "factors": [\n    [\n     "pow",\n     0\n    ],\n    [\n     "pow",\n     1\n    ]\n   ],\n   "slots": 2,\n   "theta": [\n    1.490878964786006,\n    2.486244139081395\n   ],\n   "free": [\n    false,\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T12",\n   "factors": [\n    [\n     "pow",\n     0\n    ],\n    [\n     "pow",\n     1\n    ]\n   ],\n   "slots": 2,\n   "theta": [\n    2.0,\n    5.999999999995549\n   ],\n   "free": [\n    false,\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T13",\n   "factors": [\n    [\n     "pow",\n     0\n    ],\n    [\n     "pow",\n     1\n    ]\n   ],\n   "slots": 2,\n   "theta": [\n    1.0,\n    4.9111354653321415\n   ],\n   "free": [\n    false,\n    false\n   ],\n   "group": false\n  },\n  {\n   "name": "T14",\n   "factors": [\n    [\n     "pow",\n     0\n    ]\n   ],\n   "slots": 1,\n   "theta": [\n    4.000000000000529\n   ],\n   "free": [\n    false\n   ],\n   "group": false\n  }\n ],\n "source": "warm-up on the 25 public games"\n}'}
os.makedirs('/kaggle/working/arc_agent', exist_ok=True)
for name, text in FILES.items():
    with open(os.path.join('/kaggle/working/arc_agent', name), 'w', encoding='utf-8') as f:
        f.write(text)
with open('/tmp/my_agent.py', 'w', encoding='utf-8') as f:
    f.write('"""The arc agent in the ARC-AGI-3-Agents framework (generated by arc_agent/kaggle/build.py; do not edit)."""\nimport sys\n\nsys.path.insert(0, "/kaggle/working")                               # where the notebook wrote the arc_agent package\n\nfrom arcengine import GameAction, GameState  # noqa: E402\nfrom agents.agent import Agent  # noqa: E402\nfrom arc_agent.agent import ArcAgent  # noqa: E402\n\n\nclass MyAgent(Agent):\n    """The framework runs one MyAgent per game, each in its own thread."""\n    MAX_ACTIONS = 3000\n\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n        self.core = ArcAgent(self.game_id)\n        self.pending = None\n\n    def is_done(self, frames, latest_frame):\n        return latest_frame.state is GameState.WIN\n\n    def choose_action(self, frames, latest_frame):\n        a, data = self.core.act(latest_frame)\n        self.pending = data\n        return GameAction.from_id(a)\n\n    def do_action_request(self, action):\n        """The click\'s coordinates are passed per agent, not stored on the shared GameAction enum (several games run\n        at once in threads, so the enum\'s own data could be overwritten by another game\'s click)."""\n        raw = self.arc_env.step(action, data=self.pending or {})\n        return self._convert_raw_frame_data(raw)\n')
print('wrote the arc agent:', sorted(FILES))


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /tmp/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    'random': Random,
    'myagent': MyAgent,
}
""")
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score']
                 ).to_parquet('/kaggle/working/submission.parquet', index=False)
